In [1]:
import os
os.environ["GLOG_minloglevel"] = "2"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

os.environ["XLA_CLIENT_MEM_FRACTION"] = "0.2"

from pathlib import Path

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import threading

from temgym_core.components import (
    Lens, Stigmator, Detector, Plane, SigmoidAperture, DoubleDeflector,
    sample_interpolant,
 )
from temgym_core.ray import Ray
from temgym_core.gaussian import make_gaussian
from temgym_core.run import run_to_end, run_to_end_vmapped, run_iter_vmapped
from temgym_core.evaluate import evaluate_gaussians_gpu_kernel_wrapper, evaluate_gaussians_cuda_gpu_kernel_wrapper_opt
from temgym_core.plotting import plot_model_plotly
from temgym_core.microscope_model import LensConfig, OperatingMode, MicroscopeModel
from temgym_core.config_loader import load_microscope_config
from temgym_core.constants import compute_Rc_from_voltage
from temgym_core.utils import fibonacci_spiral

import jax
import jax.numpy as jnp
import optimistix as optx

jax.config.update("jax_enable_x64", True)


In [2]:
toml_path = Path("microscope.toml")
npz_path = Path("data/microscope_design_curves.npz")


def build_model_from_toml(config_path: Path):
    cfg = load_microscope_config(config_path)

    beam_cfg = cfg.get("beam", {})
    voltage = float(beam_cfg.get("voltage_kV", 200.0)) * 1e3
    rc_default = float(compute_Rc_from_voltage(voltage))

    components_cfg = cfg.get("microscope", {}).get("components", {})
    lens_names = list(components_cfg.get("lenses", []))
    if not lens_names:
        raise ValueError("No lenses found in microscope.components.lenses")

    lens_table = cfg.get("lenses", {})
    lenses = []
    for lens_name in lens_names:
        spec = lens_table[lens_name]
        lenses.append(
            LensConfig(
                name=lens_name,
                z_position=float(spec["z_m"]),
                turns=float(spec["turns"]),
                Gc=float(spec["Gc"]),
                Rc=float(spec.get("Rc", rc_default)),
                Tc=float(spec.get("Tc", 0.0)),
            )
        )

    modes_cfg = cfg.get("modes", {})
    mode_control_labels = {}
    modes = {}
    for mode_name, mode_cfg in modes_cfg.items():
        current_lenses = list(mode_cfg.get("current_lenses", lens_names))
        current_index = {name: i for i, name in enumerate(current_lenses)}
        if any(name not in current_index for name in lens_names):
            raise ValueError(
                f"Mode '{mode_name}' current_lenses does not cover all configured lenses"
            )
        order_idx = [current_index[name] for name in lens_names]

        points = list(mode_cfg.get("points", []))
        if not points:
            raise ValueError(f"Mode '{mode_name}' has no points")

        control_values = np.asarray(
            [float(p["control_value"]) for p in points],
            dtype=float,
        )
        currents = np.asarray([p["currents_A"] for p in points], dtype=float)
        currents = currents[:, order_idx]

        full_scale_current = max(float(np.max(np.abs(currents))), 1.0)
        normalized_currents = currents / full_scale_current

        modes[mode_name] = OperatingMode(
            control_values=control_values,
            normalized_currents=normalized_currents,
            full_scale_current=full_scale_current,
            allow_signed_currents=True,
        )
        mode_control_labels[mode_name] = str(mode_cfg.get("control_name", mode_name))

    if not modes:
        raise ValueError("No operating modes found in config")

    auxiliary = {
        "config_source": "toml",
        "config_path": str(config_path),
        "z_source": 0.0,
        "gui_grid_pixels": 256,
    }

    apertures_cfg = cfg.get("apertures", {})
    aperture_names = list(components_cfg.get("apertures", []))
    if aperture_names:
        ap_name = aperture_names[0]
        ap_cfg = apertures_cfg.get(ap_name, {})
        if "z_m" in ap_cfg:
            auxiliary["z_aperture"] = float(ap_cfg["z_m"])

    deflectors_cfg = cfg.get("deflectors", {})
    deflector_names = list(components_cfg.get("deflectors", []))
    if deflector_names:
        def_name = deflector_names[0]
        def_cfg = deflectors_cfg.get(def_name, {})
        if "spacing_m" in def_cfg:
            auxiliary["deflector_spacing"] = float(def_cfg["spacing_m"]
            )
        auxiliary["deflector_drive_range"] = [-0.1, 0.1]
        auxiliary["deflector_balance_range"] = [0.2, 3.0]

    source_waist_m = float(beam_cfg.get("virtual_source_diameter_nm", 10.0)) * 1e-9
    auxiliary["source_waist_m"] = source_waist_m

    model = MicroscopeModel(
        voltage=voltage,
        lenses=tuple(lenses),
        modes=modes,
        reference_voltage=voltage,
        auxiliary=auxiliary,
    )
    return model, cfg, mode_control_labels


mode_control_labels = {}
if toml_path.exists():
    model, config_dict, mode_control_labels = build_model_from_toml(toml_path)
    model_source = f"TOML ({toml_path})"
elif npz_path.exists():
    model = MicroscopeModel.from_npz(npz_path)
    config_dict = {}
    model_source = f"NPZ ({npz_path})"
else:
    raise FileNotFoundError(
        f"Could not find {toml_path} or {npz_path}."
    )

aux = dict(getattr(model, "auxiliary", {}) or {})
source_waist_m = float(aux.get("source_waist_m", 1e-7))
z_source = float(aux.get("z_source", 0.0))

mode_names = list(model.modes.keys())
if not mode_names:
    raise ValueError("Microscope model has no operating modes")

spot_mode_name = "spot" if "spot" in model.modes else mode_names[0]
if "mag" in model.modes:
    mag_mode_name = "mag"
elif "magnification" in model.modes:
    mag_mode_name = "magnification"
else:
    mag_mode_name = mode_names[-1]

if "spot" not in model.modes:
    model.modes["spot"] = model.modes[spot_mode_name]
if "mag" not in model.modes:
    model.modes["mag"] = model.modes[mag_mode_name]

SPOT_CONTROL_LABEL = mode_control_labels.get(spot_mode_name, "Spot")
MAG_CONTROL_LABEL = mode_control_labels.get(mag_mode_name, "Mag")

spot_values = np.asarray(model.modes["spot"].control_values, dtype=float)
mag_values = np.asarray(model.modes["mag"].control_values, dtype=float)

print(f"Loaded MicroscopeModel at {model.voltage/1000:.0f} kV from {model_source}")
print(f"Modes: {list(model.modes.keys())}")
print(
    f"UI controls -> Spot: {spot_mode_name} ({SPOT_CONTROL_LABEL}), "
    f"Mag: {mag_mode_name} ({MAG_CONTROL_LABEL})"
)
print(f"Source waist: {source_waist_m*1e9:.2f} nm at z = {z_source:.4f} m")


Loaded MicroscopeModel at 200 kV from TOML (microscope.toml)
Modes: ['magnification', 'spot', 'mag']
UI controls -> Spot: magnification (system_magnification), Mag: magnification (system_magnification)
Source waist: 10.00 nm at z = 0.0000 m


In [3]:
# ── Helper functions ──────────────────────────────────────────────────



GRID_PIXELS = int(aux.get('gui_grid_pixels', 256))

N_SOURCE_RAYS = 1_000_000

FAN_NUM_RAYS = 101

APERTURE_SHARPNESS = 1e14

APERTURE_RADII_UM = [5, 10, 15, 20, 30, 50, 75, 100, 150]

# Fixed source half-angle large enough to overfill any aperture in the list

SOURCE_THETA_RAD = 1e-3



# Deflector defaults from auxiliary dict

DEFLECTOR_SPACING = float(aux.get('deflector_spacing', 0.005))

_def_range = aux.get('deflector_drive_range', [-0.1, 0.1])

DEFLECTOR_DRIVE_RANGE = [min(_def_range[0], -0.1), max(_def_range[1], 0.1)]

DEFLECTOR_BALANCE_RANGE = aux.get('deflector_balance_range', [0.2, 3.0])



# Stigmator defaults from auxiliary dict.

# Using np.inf keeps stigmators effectively disabled unless set by aux/UI.

STIGMATOR_FOCAL_RANGE_M = aux.get('stigmator_focal_range_m', [-1.0, 1.0])

COND_STIG_FX_DEFAULT = float(aux.get('condenser_stigmator_fx_m', np.inf))

COND_STIG_FY_DEFAULT = float(aux.get('condenser_stigmator_fy_m', np.inf))

OBJ_POST_STIG_FX_DEFAULT = float(aux.get('objective_post_stigmator_fx_m', np.inf))

OBJ_POST_STIG_FY_DEFAULT = float(aux.get('objective_post_stigmator_fy_m', np.inf))





def make_point_source(n, theta_max, waist, voltage, z0=0.0, randomize=False, rng=None):

    """Point source at (0,0) with E(0,0) = 1, flat wavefront, pathlength = 0."""

    n = int(n)

    theta_max = float(theta_max)

    if randomize:

        # Uniform area sampling over the source angular disk.

        if rng is None:

            rng = np.random.default_rng()

        r = theta_max * np.sqrt(rng.random(n))

        phi = 2.0 * np.pi * rng.random(n)

        dx = r * np.cos(phi)

        dy = r * np.sin(phi)

    else:

        dx, dy = fibonacci_spiral(n, theta_max)

    dx = np.asarray(dx, dtype=float)

    dy = np.asarray(dy, dtype=float)

    zeros = np.zeros(n, dtype=float)

    return make_gaussian(

        x=zeros, y=zeros, dx=dx, dy=dy,

        z=np.full(n, float(z0), dtype=float),

        voltage=np.full(n, float(voltage), dtype=float),

        waist_x=np.full(n, float(waist), dtype=float),

        waist_y=np.full(n, float(waist), dtype=float),

        amp=np.full(n, 1.0 / max(n, 1), dtype=float),

        phase=zeros,

        rcurv_x=np.full(n, np.inf, dtype=float),

        rcurv_y=np.full(n, np.inf, dtype=float),

        wavelength_unit='m',

    )





def em_to_lens(em):

    return Lens(z=float(em.z), focal_length=float(em.focal_length))





def make_detector(z, half_width, shape=GRID_PIXELS):

    pixel = (2.0 * float(half_width)) / float(shape)

    return Detector(z=float(z), pixel_size=(pixel, pixel), shape=(int(shape), int(shape)))





def intensity_image(beam, detector):

    field = np.asarray(evaluate_gaussians_cuda_gpu_kernel_wrapper_opt(beam, detector))

    return np.abs(field) ** 2





def _abcd_free(d):

    return np.array([[1.0, float(d)], [0.0, 1.0]])





def _abcd_lens(f):

    return np.array([[1.0, 0.0], [-1.0 / float(f), 1.0]])





def _abcd_stigmator(fx, fy):

    # For scalar ABCD estimates in the viewer we use harmonic-mean focal length.

    # Full anisotropy is still simulated exactly by the Stigmator component itself.

    inv_fx = 0.0 if not np.isfinite(fx) else 1.0 / float(fx)

    inv_fy = 0.0 if not np.isfinite(fy) else 1.0 / float(fy)

    inv_f = 0.5 * (inv_fx + inv_fy)

    if abs(inv_f) < 1e-30:

        return np.eye(2)

    return _abcd_lens(1.0 / inv_f)





def abcd_source_to_z(components, z_target, z_src=0.0):

    """Build ABCD matrix from z_src through components up to z_target."""

    M = np.eye(2)

    z_prev = float(z_src)

    for c in components:

        z_c = float(getattr(c, 'z'))

        if z_c > z_target + 1e-12:

            break

        M = _abcd_free(z_c - z_prev) @ M

        z_prev = z_c

        if isinstance(c, Lens):

            M = _abcd_lens(c.focal_length) @ M

        elif isinstance(c, Stigmator):

            M = _abcd_stigmator(c.focal_length_x, c.focal_length_y) @ M

    # final drift to z_target

    if z_prev < z_target - 1e-12:

        M = _abcd_free(z_target - z_prev) @ M

    return M





def beam_extent_at_z(components, theta_max, z_target, z_src=0.0):

    """Estimate beam half-width at z_target from ABCD.



    Returns |B| * theta_max (the extent of outermost ray from on-axis source).

    """

    M = abcd_source_to_z(components, z_target, z_src)

    return abs(M[0, 1]) * float(theta_max)





# ── Sample objects ────────────────────────────────────────────────────



SAMPLE_WIDTH_M = 10e-6





def smiley(size):

    '''

    Smiley face test object from https://doi.org/10.1093/micmic/ozad021

    '''

    obj = np.ones((size, size), dtype=np.complex64)

    y, x = np.ogrid[-size // 2:size // 2, -size // 2:size // 2]



    outline = (((y * 1.2) ** 2 + x**2) > (110 / 256 * size) ** 2) & (

        ((y * 1.2) ** 2 + x**2) < (120 / 256 * size) ** 2

    )

    obj[outline] = 0.0



    left_eye = ((y + 40 / 256 * size) ** 2 + (x + 40 / 256 * size) ** 2) < (20 / 256 * size) ** 2

    obj[left_eye] = 0

    right_eye = (np.abs(y + 40 / 256 * size) < 15 / 256 * size) & (

        np.abs(x - 40 / 256 * size) < 30 / 256 * size

    )

    obj[right_eye] = 0



    nose = (y + 20 / 256 * size + x > 0) & (x < 0) & (y < 10 / 256 * size)

    obj[nose] = (0.05j * x + 0.05j * y)[nose]



    mouth = (((y * 1) ** 2 + x**2) > (50 / 256 * size) ** 2) & (

        ((y * 1) ** 2 + x**2) < (70 / 256 * size) ** 2

    ) & (y > 20 / 256 * size)

    obj[mouth] = 0



    tongue = (((y - 50 / 256 * size) ** 2 + (x - 50 / 256 * size) ** 2) < (20 / 256 * size) ** 2) & (

        (y**2 + x**2) > (70 / 256 * size) ** 2

    )

    obj[tongue] = 0



    signature_wave = np.exp(1j * (3 * y + 7 * x) * 2 * np.pi / size)

    obj += 0.3 * signature_wave - 0.3

    return obj





def axes_from_width(size, width_m):

    step = width_m / size

    coords = (np.arange(size) - 0.5 * (size - 1)) * step

    return coords, coords





def build_sample(name, z, width_m=SAMPLE_WIDTH_M, grid_size=256):

    """Return a component for the sample plane.



    'None'   -> transparent Plane (no effect on beam)

    'Smiley' -> InterpolatedSample2D smiley face test object

    """

    if name == 'None':

        return Plane(z=float(z))

    elif name == 'Smiley':

        x_coords, y_coords = axes_from_width(grid_size, float(width_m))

        img = smiley(grid_size)

        amp_raw = np.abs(img)

        amp = amp_raw / (np.max(amp_raw) + 1e-12)

        phase = 1e-12 * np.angle(img)

        sample_complex = amp * np.exp(1j * phase)

        return sample_interpolant(

            sample=sample_complex.T,  # interpax expects f[x_idx, y_idx]; numpy arrays are [y_idx, x_idx]

            x_coords=x_coords,

            y_coords=y_coords,

            z=float(z),

            method='cubic',

            extrap=1.0,

        )

    else:

        return Plane(z=float(z))





def build_microscope(spot_val, mag_val, aperture_radius_um=30.0,

                     aperture_x_um=0.0, aperture_y_um=0.0,

                     deflector_shift_x=0.0, deflector_shift_y=0.0,

                     deflector_tilt_x=0.0, deflector_tilt_y=0.0,

                     deflector_shift_balance_x=1.0, deflector_shift_balance_y=1.0,

                     deflector_tilt_balance_x=1.0, deflector_tilt_balance_y=1.0,

                     condenser_stig_fx=COND_STIG_FX_DEFAULT,

                     condenser_stig_fy=COND_STIG_FY_DEFAULT,

                     objective_post_stig_fx=OBJ_POST_STIG_FX_DEFAULT,

                     objective_post_stig_fy=OBJ_POST_STIG_FY_DEFAULT,

                     sample_name='None', sample_defocus_m=0.0):

    """Build the full component list and metadata for a given spot/mag setting."""

    spot_lenses = model.build_components('spot', float(spot_val))

    mag_lenses = model.build_components('mag', float(mag_val))



    CL1_em, CL3_em, C_mini_em, Obj_prefield_em = spot_lenses[:4]

    Obj_post_em, IL1_em, IL2_em, IL3_em, PL1_em = mag_lenses[4:]



    z_sample = float(aux.get('z_sample',

                             Obj_prefield_em.z + abs(Obj_prefield_em.focal_length)))

    d_pl1_to_det = float(aux.get('d_pl1_to_detector_m', 0.325))

    z_detector = float(aux.get('z_detector', PL1_em.z + d_pl1_to_det))



    z_aperture = float(aux.get('z_aperture',

                               0.5 * (CL3_em.z + C_mini_em.z)))

    aperture = SigmoidAperture(

        z=z_aperture,

        radius=float(aperture_radius_um) * 1e-6,

        x0=float(aperture_x_um) * 1e-6,

        y0=float(aperture_y_um) * 1e-6,

        edge_width=1e-12,

        sharpness=APERTURE_SHARPNESS,

        t_low=0.0, t_high=1.0,

    )



    z_cond_stigmator = float(

        aux.get('z_condenser_stigmator', 0.5 * (float(CL3_em.z) + z_aperture))

    )

    condenser_stigmator = Stigmator(

        z=z_cond_stigmator,

        focal_length_x=float(condenser_stig_fx),

        focal_length_y=float(condenser_stig_fy),

    )



    # DoubleDeflector placed between aperture and C_mini

    z_deflector = z_aperture + 0.5 * (float(C_mini_em.z) - z_aperture - DEFLECTOR_SPACING)

    deflector = DoubleDeflector(

        z=z_deflector,

        spacing=DEFLECTOR_SPACING,

        shift_x=float(deflector_shift_x),

        shift_y=float(deflector_shift_y),

        tilt_x=float(deflector_tilt_x),

        tilt_y=float(deflector_tilt_y),

        shift_balance_x=float(deflector_shift_balance_x),

        shift_balance_y=float(deflector_shift_balance_y),

        tilt_balance_x=float(deflector_tilt_balance_x),

        tilt_balance_y=float(deflector_tilt_balance_y),

    )



    z_objective_post_stigmator = float(

        aux.get('z_objective_post_stigmator', 0.5 * (float(Obj_post_em.z) + float(IL1_em.z)))

    )

    objective_post_stigmator = Stigmator(

        z=z_objective_post_stigmator,

        focal_length_x=float(objective_post_stig_fx),

        focal_length_y=float(objective_post_stig_fy),

    )



    sample_component = build_sample(sample_name, z_sample - float(sample_defocus_m) ) # positive slider → move toward objective (under-focus))



    components = [

        em_to_lens(CL1_em),

        em_to_lens(CL3_em),

        condenser_stigmator,

        aperture,

        deflector,

        em_to_lens(C_mini_em),

        em_to_lens(Obj_prefield_em),

        sample_component,

        em_to_lens(Obj_post_em),

        objective_post_stigmator,

        em_to_lens(IL1_em),

        em_to_lens(IL2_em),

        em_to_lens(IL3_em),

        em_to_lens(PL1_em),

        Plane(z=z_detector),

    ]



    labels = [

        "CL1", "CL3", "Cond_Stig", "Aperture", "Deflector", "C_mini", "Obj_pre", "Sample",

        "Obj_post", "ObjPost_Stig", "IL1", "IL2", "IL3", "PL1", "Detector",

    ]



    plot_components = [

        CL1_em, CL3_em, Plane(z=z_cond_stigmator), Plane(z=z_aperture),

        deflector,

        C_mini_em, Obj_prefield_em,

        Plane(z=z_sample),

        Obj_post_em, Plane(z=z_objective_post_stigmator), IL1_em, IL2_em, IL3_em, PL1_em,

        make_detector(z_detector, 15e-3, shape=GRID_PIXELS),

    ]



    return {

        'components': components,

        'plot_components': plot_components,

        'labels': labels,

        'z_sample': z_sample,

        'z_detector': z_detector,

        'z_aperture': z_aperture,

        'z_deflector': z_deflector,

        'z_condenser_stigmator': z_cond_stigmator,

        'z_objective_post_stigmator': z_objective_post_stigmator,

        'aperture_radius_m': float(aperture_radius_um) * 1e-6,

        'aperture_x_um': float(aperture_x_um),

        'aperture_y_um': float(aperture_y_um),

        'condenser_stig_fx': float(condenser_stig_fx),

        'condenser_stig_fy': float(condenser_stig_fy),

        'objective_post_stig_fx': float(objective_post_stig_fx),

        'objective_post_stig_fy': float(objective_post_stig_fy),

    }





def make_source_beam(n_rays=N_SOURCE_RAYS, randomize=False):

    """Build point-source beam with the fixed source half-angle."""

    return make_point_source(

        n=int(n_rays),

        theta_max=SOURCE_THETA_RAD,

        waist=source_waist_m,

        voltage=model.voltage,

        z0=z_source,

        randomize=bool(randomize),

    )





def propagate_steps(beam, components):

    """Return [input, after_comp0, after_comp1, ...].



    ``run_iter_vmapped`` yields two outputs per component (propagator

    then component).  We keep only the component outputs (odd indices)

    so ``beam_steps[i+1]`` is the state after ``components[i]``.

    """

    all_outputs = run_iter_vmapped(beam, components)

    steps = [beam]

    steps.extend(all_outputs[1::2])  # skip propagator intermediates

    return steps





def beam_image_at_z(beam_steps, components, z_target, half_width):

    """Evaluate intensity at an arbitrary z plane using the nearest upstream state."""

    comp_zs = [float(getattr(c, 'z')) for c in components]

    best_idx = 0

    for i, cz in enumerate(comp_zs):

        if cz <= z_target + 1e-12:

            best_idx = i + 1

    beam_state = beam_steps[min(best_idx, len(beam_steps) - 1)]

    beam_z = float(np.asarray(beam_state.z).ravel()[0])

    if abs(beam_z - z_target) > 1e-12:

        beam_state = run_to_end_vmapped(beam_state, [Plane(z=z_target)])

    det = make_detector(z_target, half_width, shape=GRID_PIXELS)

    return intensity_image(beam_state, det), det





def _ray_indices_from_beam(beam, n_lines):

    """Pick representative rays by sorting on initial dx and downsampling."""

    dx = np.asarray(beam.dx, dtype=float).ravel()

    n_total = int(dx.size)

    if n_total <= 0:

        return np.array([], dtype=int)



    n_use = int(max(1, min(int(n_lines), n_total)))

    if n_use == n_total:

        return np.arange(n_total, dtype=int)



    order = np.argsort(dx)

    pos = np.linspace(0, n_total - 1, n_use)

    return order[np.round(pos).astype(int)]





def gaussian_ray_traces_from_steps(beam_steps, labels, n_lines=FAN_NUM_RAYS, min_alpha=0.04):

    """Build Plotly ray traces from Gaussian beam states.



    Rays occluded by the aperture are dimmed using the amplitude attenuation

    applied by the aperture step, read directly from beam.amplitude.

    """

    if not beam_steps:

        return []



    ray_indices = _ray_indices_from_beam(beam_steps[0], n_lines)

    if ray_indices.size == 0:

        return []



    z_line = [float(np.asarray(state.z).ravel()[0]) for state in beam_steps]



    alpha_values = np.ones(ray_indices.size, dtype=float)

    if labels is not None and 'Aperture' in labels:

        ap_step = labels.index('Aperture') + 1

        if ap_step < len(beam_steps):

            amp = np.abs(np.asarray(beam_steps[ap_step].amplitude)).ravel()

            amp_max = amp.max()

            if amp_max > 0:

                transmission = np.clip(amp / amp_max, 0.0, 1.0)



                # Ensure some occluded rays appear in the display subset.

                n_boost = max(1, n_lines // 3)

                weak = np.argsort(transmission)[:n_boost]

                ray_indices = np.unique(np.concatenate([ray_indices, weak])).astype(int)

                if ray_indices.size > n_lines:

                    sel = np.round(np.linspace(0, ray_indices.size - 1, n_lines)).astype(int)

                    ray_indices = ray_indices[sel]



                alpha_values = min_alpha + (1.0 - min_alpha) * transmission[ray_indices]



    traces = []

    for i, (ray_idx, alpha) in enumerate(zip(ray_indices, alpha_values)):

        x_line = [float(np.asarray(state.x).ravel()[ray_idx]) for state in beam_steps]

        traces.append(go.Scatter(

            x=x_line, y=z_line, mode='lines',

            line=dict(color=f'rgba(110,110,110,{alpha:.3f})', width=1.2),

            showlegend=False, hoverinfo='skip', name=f'ray-{i}',

        ))

    return traces





def component_traces_from_model(micro_dict):

    """Create component geometry traces (without using a separate ray fan)."""

    seed_ray = Ray(

        x=np.array([0.0], dtype=float),

        y=np.array([0.0], dtype=float),

        dx=np.array([0.0], dtype=float),

        dy=np.array([0.0], dtype=float),

        z=np.array([z_source], dtype=float),

        pathlength=np.array([0.0], dtype=float),

    )

    comp_fig = plot_model_plotly(

        micro_dict['plot_components'],

        rays=seed_ray,

        solution_rays=None,

        component_labels=micro_dict['labels'],

        include_input_rays=True,

        band_mode='lines',

        ray_coordinate='x_rot',

        show_component_labels=False,

        width=600, height=700,

    )

    # plot_model_plotly always adds the seed-ray trace first; keep component lines only.

    return [go.Figure(data=[t]).data[0] for t in list(comp_fig.data)[1:] if getattr(t, 'mode', None) != 'text']





# ── Solver helpers for double deflector balance ──────────────────────



def pivot_distance(spacing_val, balance_val, eps=1e-12):

    """Distance from second deflector to pivot point."""

    if abs(float(balance_val) - 1.0) < eps:

        return np.inf

    return float(spacing_val) / (float(balance_val) - 1.0)





def bounded_logistic_to_physical(u, bounds_arr):

    b = jnp.asarray(bounds_arr, dtype=jnp.float64)

    lo, hi = b[:, 0], b[:, 1]

    return lo + (hi - lo) * jax.nn.sigmoid(u)





def bounded_logit_from_physical(x, bounds_arr):

    b = np.asarray(bounds_arr, dtype=float)

    lo, hi = b[:, 0], b[:, 1]

    s = np.clip((np.asarray(x, dtype=float) - lo) / (hi - lo), 1e-9, 1.0 - 1e-9)

    return np.log(s) - np.log1p(-s)





print("Helpers defined.")


Helpers defined.


In [ ]:
# ── Interactive microscope viewer ──────────────────────────────────────



# Ray lines are drawn directly from Gaussian beam states (no second ray trace).



RAY_COUNT_OPTIONS = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

Z_STEP_OPTIONS_UM = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]

APERTURE_STEP_OPTIONS_UM = [1.0, 5.0, 10.0, 50.0, 100.0]





def _format_um_step(step_um):

    if step_um < 0.1:

        return f"{int(round(step_um * 1000.0))} nm"

    if float(step_um).is_integer():

        return f"{int(step_um)} um"

    return f"{step_um:g} um"





# Initial build

init_spot = float(spot_values[0])

init_mag = float(mag_values[0])

init_ap_um = 30.0

init_ray_count = N_SOURCE_RAYS

micro = build_microscope(init_spot, init_mag, aperture_radius_um=init_ap_um)



beam_in = make_source_beam(init_ray_count)

beam_steps = propagate_steps(beam_in, micro['components'])



comp_zs = [float(getattr(c, 'z')) for c in micro['components']]

all_zs = sorted(set([z_source] + comp_zs))

z_min, z_max = float(min(all_zs)), float(max(all_zs))



# Start at source

init_z = z_source

init_hw = beam_extent_at_z(micro['components'], SOURCE_THETA_RAD, init_z, z_source)

init_hw = max(init_hw, source_waist_m * 10) * 1.5

init_image, init_det = beam_image_at_z(beam_steps, micro['components'], init_z, init_hw)

init_extent = np.asarray(init_det.extent) * 1e6  # um



# ── Ray diagram ──────────────────────────────────────────────────────

component_traces_init = component_traces_from_model(micro)

ray_traces_init = gaussian_ray_traces_from_steps(beam_steps, micro['labels'], n_lines=FAN_NUM_RAYS)



# ── Combined figure ──────────────────────────────────────────────────

fig = go.FigureWidget(

    make_subplots(

        rows=1, cols=2,

        column_widths=[0.40, 0.60],

        horizontal_spacing=0.14,

        subplot_titles=("Beam intensity", "Ray diagram"),

    )

)





def _extent_params(extent_um, shape):

    """Return center-origin + spacing matching matplotlib extent semantics."""

    x0, x1, y0, y1 = [float(v) for v in extent_um]

    ny, nx = int(shape[0]), int(shape[1])

    dx = (x1 - x0) / max(nx, 1)

    dy = (y1 - y0) / max(ny, 1)

    return x0 + 0.5 * dx, dx, y0 + 0.5 * dy, dy





x0_init, dx_init, y0_init, dy_init = _extent_params(init_extent, init_image.shape)



# Heatmap with non-overlapping colorbar

fig.add_trace(

    go.Heatmap(

        z=init_image.tolist(),

        x0=float(x0_init),

        dx=float(dx_init),

        y0=float(y0_init),

        dy=float(dy_init),

        zsmooth=False,

        colorscale='Inferno',

        zmin=0.0,

        zmax=max(float(np.max(init_image)), 1e-30),

        showscale=True,

        colorbar=dict(

            len=1.0,

            y=0.5,

            yanchor='middle',

            x=0.34,

            xanchor='left',

            thickness=10,

            title=dict(text='I', side='right'),

        ),

    ),

    row=1, col=1,

 )

IMAGE_TRACE_IDX = 0



# Aperture circle overlay on heatmap (hidden initially, shown near aperture z)

theta_circle = np.linspace(0, 2 * np.pi, 200)

fig.add_trace(

    go.Scatter(

        x=(micro['aperture_radius_m'] * np.cos(theta_circle) * 1e6).tolist(),

        y=(micro['aperture_radius_m'] * np.sin(theta_circle) * 1e6).tolist(),

        mode='lines',

        line=dict(color='cyan', width=2, dash='dash'),

        showlegend=False,

        hoverinfo='skip',

        visible=False,

    ),

    row=1, col=1,

 )

APERTURE_CIRCLE_IDX = 1



# Component traces (col 2)

COMP_TRACE_START = 2

for trace in component_traces_init:

    fig.add_trace(trace, row=1, col=2)

COMP_TRACE_END = len(fig.data)



# Ray traces (col 2)

RAY_TRACE_START = COMP_TRACE_END

for trace in ray_traces_init:

    fig.add_trace(trace, row=1, col=2)

RAY_TRACE_END = len(fig.data)



# z-indicator line on ray diagram

fig.add_trace(

    go.Scatter(

        x=[-15e-3, 15e-3], y=[init_z, init_z],

        mode='lines',

        line=dict(color='red', width=2, dash='dash'),

        showlegend=False, hoverinfo='skip',

    ),

    row=1, col=2,

 )

Z_LINE_IDX = len(fig.data) - 1



# Aperture indicator on ray diagram: two horizontal lines with a gap

ap_r_plot = micro['aperture_radius_m']

max_x_ray = 15e-3  # plot half-width in metres

z_ap = micro['z_aperture']

fig.add_trace(

    go.Scatter(

        x=[-max_x_ray, -ap_r_plot], y=[z_ap, z_ap],

        mode='lines',

        line=dict(color='orange', width=4),

        showlegend=False, hoverinfo='skip',

    ),

    row=1, col=2,

 )

AP_RAY_LEFT_IDX = len(fig.data) - 1

fig.add_trace(

    go.Scatter(

        x=[ap_r_plot, max_x_ray], y=[z_ap, z_ap],

        mode='lines',

        line=dict(color='orange', width=4),

        showlegend=False, hoverinfo='skip',

    ),

    row=1, col=2,

 )

AP_RAY_RIGHT_IDX = len(fig.data) - 1



# Axes

fig.update_xaxes(title_text='x [um]', title_standoff=6, row=1, col=1)

fig.update_yaxes(

    title_text='y [um]',

    title_standoff=6,

    scaleanchor='x',

    scaleratio=1,

    constrain='domain',

    row=1, col=1,

)

fig.update_xaxes(title_text='x_rot [m]', range=[-max_x_ray, max_x_ray], row=1, col=2)

fig.update_yaxes(title_text='z [m]', autorange='reversed', row=1, col=2)

fig.update_xaxes(range=[float(init_extent[0]), float(init_extent[1])], row=1, col=1)

fig.update_yaxes(range=[float(init_extent[2]), float(init_extent[3])], row=1, col=1)



# Component labels on right side

x3_domain = fig.layout.xaxis2.domain

label_x = float(x3_domain[1]) + 0.01 if x3_domain else 0.68

label_annotations = []

for name, cz in zip(micro['labels'], comp_zs):

    label_annotations.append(dict(

        x=label_x, y=cz,

        xref='paper', yref='y2',

        text=f"{name} (z={cz:.4f})",

        showarrow=False, xanchor='left', font=dict(size=10),

    ))

base_annotations = list(fig.layout.annotations or [])

fig.layout.annotations = tuple(base_annotations + label_annotations)



fig.update_layout(

    height=700, width=1440,

    margin=dict(l=40, r=170, t=50, b=20),

    plot_bgcolor='white', paper_bgcolor='white',

    uirevision='keep',

)





def _sync_colorbar_to_heatmap():

    """Match colorbar height to the actually rendered heatmap height."""

    x_dom = fig.layout.xaxis.domain

    y_dom = fig.layout.yaxis.domain

    x_rng = fig.layout.xaxis.range

    y_rng = fig.layout.yaxis.range

    if not (x_dom and y_dom and x_rng and y_rng):

        return



    m = fig.layout.margin

    plot_w = float(fig.layout.width) - float(m.l) - float(m.r)

    plot_h = float(fig.layout.height) - float(m.t) - float(m.b)

    if plot_w <= 0 or plot_h <= 0:

        return



    x_dom_frac = float(x_dom[1]) - float(x_dom[0])

    y_dom_frac = float(y_dom[1]) - float(y_dom[0])

    x_span = abs(float(x_rng[1]) - float(x_rng[0]))

    y_span = abs(float(y_rng[1]) - float(y_rng[0]))

    if x_span <= 0 or y_span <= 0:

        return



    x_dom_px = x_dom_frac * plot_w

    y_dom_px = y_dom_frac * plot_h



    px_per_unit = x_dom_px / x_span

    y_needed_px = px_per_unit * y_span

    y_drawn_px = min(y_needed_px, y_dom_px)

    y_drawn_frac = y_drawn_px / plot_h



    y_center = 0.5 * (float(y_dom[0]) + float(y_dom[1]))

    x_right = float(x_dom[1])



    fig.data[IMAGE_TRACE_IDX].colorbar.len = float(max(y_drawn_frac, 1e-6))

    fig.data[IMAGE_TRACE_IDX].colorbar.y = float(y_center)

    fig.data[IMAGE_TRACE_IDX].colorbar.yanchor = 'middle'

    fig.data[IMAGE_TRACE_IDX].colorbar.x = float(x_right + 0.008)

    fig.data[IMAGE_TRACE_IDX].colorbar.xanchor = 'left'





_sync_colorbar_to_heatmap()



# ── Widgets ──────────────────────────────────────────────────────────



snap_options = [('-- free z --', None)]

for name, cz, comp in zip(micro['labels'], comp_zs, micro['components']):

    snap_options.append((f"{name} (z={cz:.4f})", cz))

    if name == "Aperture":

        z_before_ap = cz - 1e-4  # 0.1 mm before

        snap_options.append((f"Before Aperture (z={z_before_ap:.4f})", z_before_ap))

    if name == "Obj_post":

        f_obj_snap = float(getattr(comp, 'focal_length', 0.0))

        z_obj_bfp = cz + f_obj_snap

        snap_options.append((f"Obj BFP (z={z_obj_bfp:.4f})", z_obj_bfp))



spot_slider = widgets.SelectionSlider(

    options=[float(v) for v in spot_values],

    value=init_spot, description='Spot', continuous_update=False,

)

mag_slider = widgets.SelectionSlider(

    options=[float(v) for v in mag_values],

    value=init_mag, description='Mag', continuous_update=False,

)

ray_count_dropdown = widgets.Dropdown(

    options=[(f"{v:,}", int(v)) for v in RAY_COUNT_OPTIONS],

    value=int(init_ray_count), description='Beam rays',

    layout=widgets.Layout(width='180px'),

)

sample_dropdown = widgets.Dropdown(

    options=['None', 'Smiley'],

    value='None',

    description='Sample',

    layout=widgets.Layout(width='200px'),

)

sample_defocus_slider = widgets.FloatSlider(

    value=0.0, min=-50, max=50, step=0.01,

    description='Defocus [um]', continuous_update=False,

    readout_format='.2f',

    layout=widgets.Layout(width='350px'),

)

aperture_dropdown = widgets.Dropdown(

    options=[(f"{r} um", float(r)) for r in APERTURE_RADII_UM],

    value=init_ap_um, description='Radius',

    layout=widgets.Layout(width='170px'),

)

ap_step_dropdown = widgets.Dropdown(

    options=[(_format_um_step(step_um), float(step_um)) for step_um in APERTURE_STEP_OPTIONS_UM],

    value=5.0, description='Move step',

    layout=widgets.Layout(width='170px'),

)

z_slider = widgets.FloatSlider(

    value=init_z, min=z_min, max=z_max, step=0.0001,

    description='z [m]', continuous_update=False,

    readout_format='.4f',

    layout=widgets.Layout(width='600px'),

)

snap_dropdown = widgets.Dropdown(

    options=snap_options,

    value=None, description='Snap to:',

    layout=widgets.Layout(width='260px'),

)

hw_slider = widgets.FloatLogSlider(

    value=init_hw * 1e3, base=10,

    min=-7, max=2, step=0.1,

    description='HW [mm]', continuous_update=False,

    layout=widgets.Layout(width='260px'),

)

gain_slider = widgets.FloatLogSlider(

    value=1.0, base=10, min=-3, max=6, step=0.1,

    description='Gain', continuous_update=False,

    layout=widgets.Layout(width='230px'),

)

auto_extent_toggle = widgets.Checkbox(

    value=False, description='Auto extent',

    layout=widgets.Layout(width='130px'),

)



z_step_dropdown = widgets.Dropdown(

    options=[(_format_um_step(step_um), float(step_um)) for step_um in Z_STEP_OPTIONS_UM],

    value=1.0,

    description='dz',

    layout=widgets.Layout(width='150px'),

)

z_minus_btn = widgets.Button(description='-z', layout=widgets.Layout(width='44px'))

z_plus_btn = widgets.Button(description='+z', layout=widgets.Layout(width='44px'))



# Aperture position nudge buttons

ap_x_label = widgets.Label(value=f"x = {micro['aperture_x_um']:.1f} um", layout=widgets.Layout(width='90px'))

ap_y_label = widgets.Label(value=f"y = {micro['aperture_y_um']:.1f} um", layout=widgets.Layout(width='90px'))

ap_x_minus = widgets.Button(description='-x', layout=widgets.Layout(width='44px'))

ap_x_plus = widgets.Button(description='+x', layout=widgets.Layout(width='44px'))

ap_y_minus = widgets.Button(description='-y', layout=widgets.Layout(width='44px'))

ap_y_plus = widgets.Button(description='+y', layout=widgets.Layout(width='44px'))



# ── Deflector widgets ────────────────────────────────────────────────



_dr_lo, _dr_hi = float(DEFLECTOR_DRIVE_RANGE[0]), float(DEFLECTOR_DRIVE_RANGE[1])

_bl_lo, _bl_hi = float(DEFLECTOR_BALANCE_RANGE[0]), float(DEFLECTOR_BALANCE_RANGE[1])



shift_x_slider = widgets.FloatSlider(

    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,

    description='Shift X', continuous_update=False,

    readout_format='.4f',

    layout=widgets.Layout(width='350px'),

)

shift_y_slider = widgets.FloatSlider(

    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,

    description='Shift Y', continuous_update=False,

    readout_format='.4f',

    layout=widgets.Layout(width='350px'),

)

tilt_x_slider = widgets.FloatSlider(

    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,

    description='Tilt X', continuous_update=False,

    readout_format='.4f',

    layout=widgets.Layout(width='350px'),

)

tilt_y_slider = widgets.FloatSlider(

    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,

    description='Tilt Y', continuous_update=False,

    readout_format='.4f',

    layout=widgets.Layout(width='350px'),

)

shift_balance_x_slider = widgets.FloatSlider(

    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,

    description='Shift Bal X', continuous_update=False,

    readout_format='.3f',

    layout=widgets.Layout(width='350px'),

)

shift_balance_y_slider = widgets.FloatSlider(

    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,

    description='Shift Bal Y', continuous_update=False,

    readout_format='.3f',

    layout=widgets.Layout(width='350px'),

)

tilt_balance_x_slider = widgets.FloatSlider(

    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,

    description='Tilt Bal X', continuous_update=False,

    readout_format='.3f',

    layout=widgets.Layout(width='350px'),

)

tilt_balance_y_slider = widgets.FloatSlider(

    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,

    description='Tilt Bal Y', continuous_update=False,

    readout_format='.3f',

    layout=widgets.Layout(width='350px'),

)

pivot_shift_x_label = widgets.Label(

    value=f"Shift Pivot X: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",

    layout=widgets.Layout(width='250px'),

)

pivot_shift_y_label = widgets.Label(

    value=f"Shift Pivot Y: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",

    layout=widgets.Layout(width='250px'),

)

pivot_tilt_x_label = widgets.Label(

    value=f"Tilt Pivot X: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",

    layout=widgets.Layout(width='250px'),

)

pivot_tilt_y_label = widgets.Label(

    value=f"Tilt Pivot Y: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",

    layout=widgets.Layout(width='250px'),

)



# ── Wobble widgets ───────────────────────────────────────────────────



wobble_toggle = widgets.Checkbox(

    value=False, description='Wobble',

    layout=widgets.Layout(width='100px'),

)

wobble_amp_slider = widgets.FloatSlider(

    value=1e-3, min=1e-4, max=_dr_hi, step=1e-4,

    description='Ampl', continuous_update=False,

    readout_format='.4f',

    layout=widgets.Layout(width='280px'),

)

wobble_axis_dropdown = widgets.Dropdown(

    options=['x', 'y', 'both'],

    value='x', description='Axis',

    layout=widgets.Layout(width='130px'),

)

wobble_warning = widgets.Label(value='', layout=widgets.Layout(width='300px'))



# ── Accumulation widgets ─────────────────────────────────────────────



accum_toggle = widgets.Checkbox(

    value=False, description='Accumulate',

    layout=widgets.Layout(width='120px'),

)

accum_persist_toggle = widgets.Checkbox(

    value=True, description='Persist trail',

    layout=widgets.Layout(width='130px'),

)

accum_rays_dropdown = widgets.Dropdown(

    options=[('100', 100), ('500', 500), ('1k', 1_000), ('5k', 5_000), ('10k', 10_000)],

    value=1_000,

    description='Rays/frame',

    layout=widgets.Layout(width='170px'),

)

accum_rate_dropdown = widgets.Dropdown(

    options=[('Fast (20 Hz)', 0.05), ('Medium (10 Hz)', 0.10), ('Slow (5 Hz)', 0.20)],

    value=0.05,

    description='Update',

    layout=widgets.Layout(width='200px'),

)

accum_fade_slider = widgets.FloatSlider(

    value=0.98, min=0.80, max=1.00, step=0.01,

    description='Retain', readout_format='.2f',

    layout=widgets.Layout(width='280px'),

)

accum_clear_btn = widgets.Button(

    description='Clear',

    layout=widgets.Layout(width='70px'),

)

accum_label = widgets.Label(value='0 frames · 0 rays', layout=widgets.Layout(width='260px'))



# ── Solver widgets ───────────────────────────────────────────────────



solve_shift_balance_btn = widgets.Button(

    description='Solve Shift Bal',

    button_style='primary',

    layout=widgets.Layout(width='150px'),

)

solve_tilt_balance_btn = widgets.Button(

    description='Solve Tilt Bal',

    button_style='primary',

    layout=widgets.Layout(width='150px'),

)

solve_status_label = widgets.Label(value='', layout=widgets.Layout(width='500px'))



# ── Reset widget ─────────────────────────────────────────────────────



reset_deflector_btn = widgets.Button(

    description='Reset Deflector',

    button_style='warning',

    layout=widgets.Layout(width='140px'),

)



# ── State ────────────────────────────────────────────────────────────

_cache = {

    'micro': micro,

    'beam_steps': beam_steps,

    'beam_in': beam_in,

    'ap_x_um': float(micro['aperture_x_um']),

    'ap_y_um': float(micro['aperture_y_um']),

    'deflector_shift_x': 0.0,

    'deflector_shift_y': 0.0,

    'deflector_tilt_x': 0.0,

    'deflector_tilt_y': 0.0,

    'deflector_shift_balance_x': 1.0,

    'deflector_shift_balance_y': 1.0,

    'deflector_tilt_balance_x': 1.0,

    'deflector_tilt_balance_y': 1.0,

}



# Wobble state

_wobble_state = {

    'timer': None,

    'phase': 0.0,

}



# Accumulation state

_accum_state = {

    'timer': None,

    'buffer': None,

    'det': None,

    'n_frames': 0,

}





def _accum_status_text():

    n_frames = int(_accum_state['n_frames'])

    rays_total = n_frames * int(accum_rays_dropdown.value)

    return f"{n_frames} frames · {rays_total:,} rays"





def _accum_clear(*, stop_timer=True, reset_buffer=True):

    if stop_timer:

        t = _accum_state.get('timer')

        if t is not None:

            t.cancel()

            _accum_state['timer'] = None

    if reset_buffer:

        _accum_state['buffer'] = None

        _accum_state['det'] = None

        _accum_state['n_frames'] = 0

    accum_label.value = _accum_status_text()





def _compute_half_width(micro_now, z_target):

    if auto_extent_toggle.value:

        hw_est = beam_extent_at_z(micro_now['components'], SOURCE_THETA_RAD, z_target, z_source)

        hw_est = max(hw_est, source_waist_m * 5) * 1.5

        hw = hw_est

        hw_slider.unobserve(update_beam_image, names='value')

        hw_slider.value = hw * 1e3

        hw_slider.observe(update_beam_image, names='value')

    else:

        hw = float(hw_slider.value) * 1e-3

    return float(hw)





def _accum_tick():

    if not accum_toggle.value:

        return



    micro_now = _cache['micro']

    z_target = float(z_slider.value)

    n_batch = int(accum_rays_dropdown.value)



    hw = _compute_half_width(micro_now, z_target)

    beam_new = make_source_beam(n_batch, randomize=True)

    steps_new = propagate_steps(beam_new, micro_now['components'])

    img_new, det_new = beam_image_at_z(steps_new, micro_now['components'], z_target, hw)



    if _accum_state['buffer'] is None or _accum_state['det'] is None:

        _accum_state['buffer'] = np.asarray(img_new, dtype=np.float64)

        _accum_state['det'] = det_new

        _accum_state['n_frames'] = 1

    else:

        det_prev = _accum_state['det']

        same_shape = tuple(getattr(det_prev, 'shape', ())) == tuple(getattr(det_new, 'shape', ()))

        same_extent = np.allclose(np.asarray(det_prev.extent, dtype=float), np.asarray(det_new.extent, dtype=float), rtol=1e-12, atol=1e-15)

        if same_shape and same_extent:

            _accum_state['buffer'] = _accum_state['buffer'] + np.asarray(img_new, dtype=np.float64)

            _accum_state['n_frames'] = int(_accum_state['n_frames']) + 1

        else:

            _accum_state['buffer'] = np.asarray(img_new, dtype=np.float64)

            _accum_state['det'] = det_new

            _accum_state['n_frames'] = 1



    accum_label.value = _accum_status_text()

    update_beam_image()



    if accum_toggle.value:

        t = threading.Timer(0.10, _accum_tick)

        t.daemon = True

        _accum_state['timer'] = t

        t.start()





def _on_accum_toggle(change):

    if change['new']:

        _accum_clear(stop_timer=True, reset_buffer=True)

        _accum_tick()

    else:

        _accum_clear(stop_timer=True, reset_buffer=True)

        update_beam_image()





def _on_accum_clear(_):

    _accum_clear(stop_timer=True, reset_buffer=True)

    if accum_toggle.value:

        _accum_tick()

    else:

        update_beam_image()





def _on_z_change(*_):

    if accum_toggle.value:

        _accum_clear(stop_timer=True, reset_buffer=True)

        _accum_tick()

    else:

        update_beam_image()





def _update_snap_options(labels, comp_zs_new, components=None):

    opts = [('-- free z --', None)]

    for idx, (name, cz) in enumerate(zip(labels, comp_zs_new)):

        opts.append((f"{name} (z={cz:.4f})", cz))

        if name == "Aperture":

            z_before = cz - 1e-4

            opts.append((f"Before Aperture (z={z_before:.4f})", z_before))

        if name == "Obj_post" and components is not None:

            f_obj_snap = float(getattr(components[idx], 'focal_length', 0.0))

            z_obj_bfp = cz + f_obj_snap

            opts.append((f"Obj BFP (z={z_obj_bfp:.4f})", z_obj_bfp))

    snap_dropdown.options = opts





def _update_aperture_labels():

    ap_x_label.value = f"x = {_cache['ap_x_um']:.1f} um"

    ap_y_label.value = f"y = {_cache['ap_y_um']:.1f} um"





def _update_pivot_labels():

    sbx = _cache['deflector_shift_balance_x']

    sby = _cache['deflector_shift_balance_y']

    tbx = _cache['deflector_tilt_balance_x']

    tby = _cache['deflector_tilt_balance_y']

    psx = pivot_distance(DEFLECTOR_SPACING, sbx)

    psy = pivot_distance(DEFLECTOR_SPACING, sby)

    ptx = pivot_distance(DEFLECTOR_SPACING, tbx)

    pty = pivot_distance(DEFLECTOR_SPACING, tby)

    pivot_shift_x_label.value = (

        "Shift Pivot X: inf (parallel shift)" if np.isinf(psx)

        else f"Shift Pivot X: {psx*1e3:.2f} mm from def2"

    )

    pivot_shift_y_label.value = (

        "Shift Pivot Y: inf (parallel shift)" if np.isinf(psy)

        else f"Shift Pivot Y: {psy*1e3:.2f} mm from def2"

    )

    pivot_tilt_x_label.value = (

        "Tilt Pivot X: inf (parallel shift)" if np.isinf(ptx)

        else f"Tilt Pivot X: {ptx*1e3:.2f} mm from def2"

    )

    pivot_tilt_y_label.value = (

        "Tilt Pivot Y: inf (parallel shift)" if np.isinf(pty)

        else f"Tilt Pivot Y: {pty*1e3:.2f} mm from def2"

    )





def _aperture_step_um():

    return float(ap_step_dropdown.value)





def _update_aperture_on_ray_diagram(micro_dict):

    ap_r = micro_dict['aperture_radius_m']

    z_ap_local = micro_dict['z_aperture']

    fig.data[AP_RAY_LEFT_IDX].x = [-max_x_ray, -ap_r]

    fig.data[AP_RAY_LEFT_IDX].y = [z_ap_local, z_ap_local]

    fig.data[AP_RAY_RIGHT_IDX].x = [ap_r, max_x_ray]

    fig.data[AP_RAY_RIGHT_IDX].y = [z_ap_local, z_ap_local]





def rebuild_microscope(*_):

    _cache['deflector_shift_x'] = float(shift_x_slider.value)

    _cache['deflector_shift_y'] = float(shift_y_slider.value)

    _cache['deflector_tilt_x'] = float(tilt_x_slider.value)

    _cache['deflector_tilt_y'] = float(tilt_y_slider.value)

    _cache['deflector_shift_balance_x'] = float(shift_balance_x_slider.value)

    _cache['deflector_shift_balance_y'] = float(shift_balance_y_slider.value)

    _cache['deflector_tilt_balance_x'] = float(tilt_balance_x_slider.value)

    _cache['deflector_tilt_balance_y'] = float(tilt_balance_y_slider.value)



    micro_new = build_microscope(

        float(spot_slider.value),

        float(mag_slider.value),

        aperture_radius_um=float(aperture_dropdown.value),

        aperture_x_um=_cache['ap_x_um'],

        aperture_y_um=_cache['ap_y_um'],

        deflector_shift_x=_cache['deflector_shift_x'],

        deflector_shift_y=_cache['deflector_shift_y'],

        deflector_tilt_x=_cache['deflector_tilt_x'],

        deflector_tilt_y=_cache['deflector_tilt_y'],

        deflector_shift_balance_x=_cache['deflector_shift_balance_x'],

        deflector_shift_balance_y=_cache['deflector_shift_balance_y'],

        deflector_tilt_balance_x=_cache['deflector_tilt_balance_x'],

        deflector_tilt_balance_y=_cache['deflector_tilt_balance_y'],

        sample_name=sample_dropdown.value,

        sample_defocus_m=float(sample_defocus_slider.value) * 100e-6,

    )

    beam_in_new = make_source_beam(int(ray_count_dropdown.value))

    beam_steps_new = propagate_steps(beam_in_new, micro_new['components'])

    _cache['micro'] = micro_new

    _cache['beam_steps'] = beam_steps_new

    _cache['beam_in'] = beam_in_new



    component_traces = component_traces_from_model(micro_new)

    ray_traces = gaussian_ray_traces_from_steps(beam_steps_new, micro_new['labels'], n_lines=FAN_NUM_RAYS)



    new_comp_zs = [float(getattr(c, 'z')) for c in micro_new['components']]

    _update_snap_options(micro_new['labels'], new_comp_zs, micro_new['components'])



    new_labels = []

    for name, cz in zip(micro_new['labels'], new_comp_zs):

        new_labels.append(dict(

            x=label_x, y=cz,

            xref='paper', yref='y2',

            text=f"{name} (z={cz:.4f})",

            showarrow=False, xanchor='left', font=dict(size=10),

        ))



    with fig.batch_update():

        n_comp = min(COMP_TRACE_END - COMP_TRACE_START, len(component_traces))

        for i in range(n_comp):

            src = component_traces[i]

            fig.data[COMP_TRACE_START + i].x = list(src.x) if src.x is not None else []

            fig.data[COMP_TRACE_START + i].y = list(src.y) if src.y is not None else []

        n_ray = min(RAY_TRACE_END - RAY_TRACE_START, len(ray_traces))

        for i in range(n_ray):

            src = ray_traces[i]

            fig.data[RAY_TRACE_START + i].x = list(src.x) if src.x is not None else []

            fig.data[RAY_TRACE_START + i].y = list(src.y) if src.y is not None else []

            fig.data[RAY_TRACE_START + i].line.color = src.line.color

        fig.layout.annotations = tuple(base_annotations + new_labels)

        _update_aperture_on_ray_diagram(micro_new)



    _update_pivot_labels()

    if accum_toggle.value and not accum_persist_toggle.value:

        _accum_clear(stop_timer=True, reset_buffer=True)

        _accum_tick()

    else:

        update_beam_image()





def update_beam_image(*_):

    micro_now = _cache['micro']

    beam_steps_now = _cache['beam_steps']

    z_target = float(z_slider.value)

    gain = float(gain_slider.value)



    if accum_toggle.value and _accum_state['buffer'] is not None and _accum_state['det'] is not None:

        img = np.asarray(_accum_state['buffer'], dtype=np.float64)

        det = _accum_state['det']

    else:

        hw = _compute_half_width(micro_now, z_target)

        img, det = beam_image_at_z(beam_steps_now, micro_now['components'], z_target, hw)



    extent = np.asarray(det.extent) * 1e6  # um

    if accum_toggle.value and int(_accum_state['n_frames']) > 0:

        img_display = img / float(_accum_state['n_frames'])

    else:

        img_display = img

    scaled = img_display * gain



    z_ap_local = micro_now['z_aperture']

    show_circle = abs(z_target - z_ap_local) < 2e-3

    ap_r_um = micro_now['aperture_radius_m'] * 1e6

    ap_x_um = _cache['ap_x_um']

    ap_y_um = _cache['ap_y_um']



    with fig.batch_update():

        x0_hm, dx_hm, y0_hm, dy_hm = _extent_params(extent, img.shape)

        fig.data[IMAGE_TRACE_IDX].z = scaled.tolist()

        fig.data[IMAGE_TRACE_IDX].x0 = float(x0_hm)

        fig.data[IMAGE_TRACE_IDX].dx = float(dx_hm)

        fig.data[IMAGE_TRACE_IDX].y0 = float(y0_hm)

        fig.data[IMAGE_TRACE_IDX].dy = float(dy_hm)

        fig.data[IMAGE_TRACE_IDX].zmax = max(float(np.max(scaled)), 1e-30)

        fig.layout.xaxis.range = [float(extent[0]), float(extent[1])]

        fig.layout.yaxis.range = [float(extent[2]), float(extent[3])]



        _sync_colorbar_to_heatmap()



        fig.data[Z_LINE_IDX].y = [z_target, z_target]



        fig.data[APERTURE_CIRCLE_IDX].visible = show_circle

        if show_circle:

            fig.data[APERTURE_CIRCLE_IDX].x = (ap_x_um + ap_r_um * np.cos(theta_circle)).tolist()

            fig.data[APERTURE_CIRCLE_IDX].y = (ap_y_um + ap_r_um * np.sin(theta_circle)).tolist()





def on_snap(change):

    val = change['new']

    if val is not None:

        z_slider.value = float(val)





def _nudge_z(direction):

    dz_m = float(z_step_dropdown.value) * 1e-6

    z_new = float(np.clip(float(z_slider.value) + direction * dz_m, float(z_slider.min), float(z_slider.max)))

    z_slider.value = z_new





def _nudge_aperture(dx_um, dy_um):

    step_um = _aperture_step_um()

    _cache['ap_x_um'] += dx_um * step_um

    _cache['ap_y_um'] += dy_um * step_um

    _update_aperture_labels()

    rebuild_microscope()





# ── Wobble logic ─────────────────────────────────────────────────────



def _wobble_tick():

    """One wobble tick: oscillate drive, rebuild, schedule next."""

    if not wobble_toggle.value:

        return

    amp = float(wobble_amp_slider.value)

    axis = wobble_axis_dropdown.value

    _wobble_state['phase'] += 0.15  # ~15% of a cycle per tick

    phase = _wobble_state['phase']

    sin_val = float(np.sin(2.0 * np.pi * phase))

    cos_val = float(np.cos(2.0 * np.pi * phase))



    # Unobserve to avoid double-rebuild

    tilt_x_slider.unobserve(rebuild_microscope, names='value')

    tilt_y_slider.unobserve(rebuild_microscope, names='value')

    if axis == 'x':

        tilt_x_slider.value = float(np.clip(amp * sin_val, _dr_lo, _dr_hi))

    elif axis == 'y':

        tilt_y_slider.value = float(np.clip(amp * sin_val, _dr_lo, _dr_hi))

    else:  # both

        tilt_x_slider.value = float(np.clip(amp * sin_val, _dr_lo, _dr_hi))

        tilt_y_slider.value = float(np.clip(amp * cos_val, _dr_lo, _dr_hi))

    tilt_x_slider.observe(rebuild_microscope, names='value')

    tilt_y_slider.observe(rebuild_microscope, names='value')



    rebuild_microscope()



    if wobble_toggle.value:

        t = threading.Timer(0.25, _wobble_tick)

        t.daemon = True

        _wobble_state['timer'] = t

        t.start()





def _on_wobble_toggle(change):

    if change['new']:

        wobble_warning.value = '⚠ Use low ray count for wobble'

        _wobble_state['phase'] = 0.0

        _wobble_tick()

    else:

        wobble_warning.value = ''

        t = _wobble_state.get('timer')

        if t is not None:

            t.cancel()

            _wobble_state['timer'] = None





# ── Solver logic ─────────────────────────────────────────────────────



def _solver_context():

    micro_now = _cache['micro']

    comps_now = list(micro_now['components'])

    labels_now = list(micro_now['labels'])

    def_idx = labels_now.index('Deflector')

    sample_idx = labels_now.index('Sample')

    def_comp = comps_now[def_idx]

    return comps_now, def_idx, sample_idx, float(def_comp.z), float(def_comp.spacing)





def _sample_state_for_solver(shift_xy, tilt_xy, shift_bal, tilt_bal):

    comps_base, def_idx, sample_idx, z_def, d_def = _solver_context()

    deflector = DoubleDeflector(

        z=z_def,

        spacing=d_def,

        shift_x=shift_xy[0],

        shift_y=shift_xy[1],

        tilt_x=tilt_xy[0],

        tilt_y=tilt_xy[1],

        shift_balance_x=shift_bal[0],

        shift_balance_y=shift_bal[1],

        tilt_balance_x=tilt_bal[0],

        tilt_balance_y=tilt_bal[1],

    )

    comps = list(comps_base)

    comps[def_idx] = deflector

    ray0 = Ray(

        x=jnp.array(0.0), y=jnp.array(0.0),

        dx=jnp.array(0.0), dy=jnp.array(0.0),

        z=jnp.array(z_source),

        pathlength=jnp.array(0.0),

    )

    out = run_to_end(ray0, comps[:sample_idx + 1])

    return jnp.stack([out.x, out.y, out.dx, out.dy])





def _solve_balance_2d(initial_phys, residual_physical):

    bounds_arr = jnp.asarray([[_bl_lo, _bl_hi], [_bl_lo, _bl_hi]], dtype=jnp.float64)

    lo, hi = bounds_arr[:, 0], bounds_arr[:, 1]

    x0_phys = jnp.clip(jnp.asarray(initial_phys, dtype=jnp.float64), lo + 1e-6, hi - 1e-6)

    s0 = jnp.clip((x0_phys - lo) / (hi - lo), 1e-9, 1.0 - 1e-9)

    u0 = jnp.log(s0) - jnp.log1p(-s0)



    def residual_unbounded(u, args):

        ba, = args

        phys = bounded_logistic_to_physical(u, ba)

        return residual_physical(phys)



    solver = optx.LevenbergMarquardt(rtol=1e-10, atol=1e-10)

    sol = optx.root_find(

        residual_unbounded,

        solver,

        y0=u0,

        args=(bounds_arr,),

        options={'jac': 'fwd'},

        max_steps=200,

        throw=False,

    )



    u_star = jnp.asarray(sol.value, dtype=jnp.float64)

    x_star = bounded_logistic_to_physical(u_star, bounds_arr)

    res = residual_physical(x_star)

    return np.asarray(x_star, dtype=float), float(jnp.linalg.norm(res)), sol





def _on_solve_shift_balance(btn):

    solve_status_label.value = 'Solving shift balance...'

    solve_shift_balance_btn.disabled = True

    solve_tilt_balance_btn.disabled = True

    try:

        tilt_bal_fixed = jnp.asarray([

            _cache['deflector_tilt_balance_x'],

            _cache['deflector_tilt_balance_y'],

        ], dtype=jnp.float64)



        def residual_shift(shift_bal_xy):

            shift_xy0 = jnp.zeros(2, dtype=jnp.float64)

            tilt_xy0 = jnp.zeros(2, dtype=jnp.float64)

            J = jax.jacfwd(

                lambda sxy: _sample_state_for_solver(sxy, tilt_xy0, shift_bal_xy, tilt_bal_fixed)

            )(shift_xy0)

            return jnp.asarray([J[2, 0], J[3, 1]], dtype=jnp.float64)



        x_star, res_norm, sol = _solve_balance_2d(

            [_cache['deflector_shift_balance_x'], _cache['deflector_shift_balance_y']],

            residual_shift,

        )

        sbx_sol, sby_sol = float(x_star[0]), float(x_star[1])



        shift_balance_x_slider.unobserve(rebuild_microscope, names='value')

        shift_balance_y_slider.unobserve(rebuild_microscope, names='value')

        shift_balance_x_slider.value = float(np.clip(sbx_sol, _bl_lo, _bl_hi))

        shift_balance_y_slider.value = float(np.clip(sby_sol, _bl_lo, _bl_hi))

        shift_balance_x_slider.observe(rebuild_microscope, names='value')

        shift_balance_y_slider.observe(rebuild_microscope, names='value')



        rebuild_microscope()



        psx = pivot_distance(DEFLECTOR_SPACING, sbx_sol)

        psy = pivot_distance(DEFLECTOR_SPACING, sby_sol)

        steps = sol.stats.get('num_steps', '?') if hasattr(sol, 'stats') else '?'

        result = getattr(sol, 'result', '?')

        solve_status_label.value = (

            f"Shift solved. bal=({sbx_sol:.4f}, {sby_sol:.4f}) "

            f"pivot=({psx*1e3:.2f} mm, {psy*1e3:.2f} mm) "

            f"|res|={res_norm:.2e} steps={steps} result={result}"

        )

    except Exception as e:

        solve_status_label.value = f"Shift solver error: {e}"

    finally:

        solve_shift_balance_btn.disabled = False

        solve_tilt_balance_btn.disabled = False





def _on_solve_tilt_balance(btn):

    solve_status_label.value = 'Solving tilt balance...'

    solve_shift_balance_btn.disabled = True

    solve_tilt_balance_btn.disabled = True

    try:

        shift_bal_fixed = jnp.asarray([

            _cache['deflector_shift_balance_x'],

            _cache['deflector_shift_balance_y'],

        ], dtype=jnp.float64)



        def residual_tilt(tilt_bal_xy):

            shift_xy0 = jnp.zeros(2, dtype=jnp.float64)

            tilt_xy0 = jnp.zeros(2, dtype=jnp.float64)

            J = jax.jacfwd(

                lambda txy: _sample_state_for_solver(shift_xy0, txy, shift_bal_fixed, tilt_bal_xy)

            )(tilt_xy0)

            return jnp.asarray([J[0, 0], J[1, 1]], dtype=jnp.float64)



        x_star, res_norm, sol = _solve_balance_2d(

            [_cache['deflector_tilt_balance_x'], _cache['deflector_tilt_balance_y']],

            residual_tilt,

        )

        tbx_sol, tby_sol = float(x_star[0]), float(x_star[1])



        tilt_balance_x_slider.unobserve(rebuild_microscope, names='value')

        tilt_balance_y_slider.unobserve(rebuild_microscope, names='value')

        tilt_balance_x_slider.value = float(np.clip(tbx_sol, _bl_lo, _bl_hi))

        tilt_balance_y_slider.value = float(np.clip(tby_sol, _bl_lo, _bl_hi))

        tilt_balance_x_slider.observe(rebuild_microscope, names='value')

        tilt_balance_y_slider.observe(rebuild_microscope, names='value')



        rebuild_microscope()



        ptx = pivot_distance(DEFLECTOR_SPACING, tbx_sol)

        pty = pivot_distance(DEFLECTOR_SPACING, tby_sol)

        steps = sol.stats.get('num_steps', '?') if hasattr(sol, 'stats') else '?'

        result = getattr(sol, 'result', '?')

        solve_status_label.value = (

            f"Tilt solved. bal=({tbx_sol:.4f}, {tby_sol:.4f}) "

            f"pivot=({ptx*1e3:.2f} mm, {pty*1e3:.2f} mm) "

            f"|res|={res_norm:.2e} steps={steps} result={result}"

        )

    except Exception as e:

        solve_status_label.value = f"Tilt solver error: {e}"

    finally:

        solve_shift_balance_btn.disabled = False

        solve_tilt_balance_btn.disabled = False





# ── Reset logic ──────────────────────────────────────────────────────



def _on_reset_deflector(btn):

    # Stop wobble

    wobble_toggle.value = False

    t = _wobble_state.get('timer')

    if t is not None:

        t.cancel()

        _wobble_state['timer'] = None



    # Unobserve all deflector sliders

    shift_x_slider.unobserve(rebuild_microscope, names='value')

    shift_y_slider.unobserve(rebuild_microscope, names='value')

    tilt_x_slider.unobserve(rebuild_microscope, names='value')

    tilt_y_slider.unobserve(rebuild_microscope, names='value')

    shift_balance_x_slider.unobserve(rebuild_microscope, names='value')

    shift_balance_y_slider.unobserve(rebuild_microscope, names='value')

    tilt_balance_x_slider.unobserve(rebuild_microscope, names='value')

    tilt_balance_y_slider.unobserve(rebuild_microscope, names='value')



    shift_x_slider.value = float(aux.get('deflector_shift_x_default', 0.0))

    shift_y_slider.value = float(aux.get('deflector_shift_y_default', 0.0))

    tilt_x_slider.value = float(aux.get('deflector_tilt_x_default', 0.0))

    tilt_y_slider.value = float(aux.get('deflector_tilt_y_default', 0.0))

    shift_balance_x_slider.value = float(aux.get('deflector_shift_balance_x_default', 1.0))

    shift_balance_y_slider.value = float(aux.get('deflector_shift_balance_y_default', 1.0))

    tilt_balance_x_slider.value = float(aux.get('deflector_tilt_balance_x_default', 1.0))

    tilt_balance_y_slider.value = float(aux.get('deflector_tilt_balance_y_default', 1.0))



    shift_x_slider.observe(rebuild_microscope, names='value')

    shift_y_slider.observe(rebuild_microscope, names='value')

    tilt_x_slider.observe(rebuild_microscope, names='value')

    tilt_y_slider.observe(rebuild_microscope, names='value')

    shift_balance_x_slider.observe(rebuild_microscope, names='value')

    shift_balance_y_slider.observe(rebuild_microscope, names='value')

    tilt_balance_x_slider.observe(rebuild_microscope, names='value')

    tilt_balance_y_slider.observe(rebuild_microscope, names='value')



    solve_status_label.value = ''

    rebuild_microscope()





# ── Wire callbacks ───────────────────────────────────────────────────



ap_x_minus.on_click(lambda _: _nudge_aperture(-1.0, 0.0))

ap_x_plus.on_click(lambda _: _nudge_aperture(+1.0, 0.0))

ap_y_minus.on_click(lambda _: _nudge_aperture(0.0, -1.0))

ap_y_plus.on_click(lambda _: _nudge_aperture(0.0, +1.0))

z_minus_btn.on_click(lambda _: _nudge_z(-1.0))

z_plus_btn.on_click(lambda _: _nudge_z(+1.0))



spot_slider.observe(rebuild_microscope, names='value')

mag_slider.observe(rebuild_microscope, names='value')

ray_count_dropdown.observe(rebuild_microscope, names='value')

aperture_dropdown.observe(rebuild_microscope, names='value')

sample_dropdown.observe(rebuild_microscope, names='value')

sample_defocus_slider.observe(rebuild_microscope, names='value')

z_slider.observe(_on_z_change, names='value')

hw_slider.observe(update_beam_image, names='value')

gain_slider.observe(update_beam_image, names='value')

auto_extent_toggle.observe(update_beam_image, names='value')

snap_dropdown.observe(on_snap, names='value')



accum_toggle.observe(_on_accum_toggle, names='value')

accum_clear_btn.on_click(_on_accum_clear)



shift_x_slider.observe(rebuild_microscope, names='value')

shift_y_slider.observe(rebuild_microscope, names='value')

tilt_x_slider.observe(rebuild_microscope, names='value')

tilt_y_slider.observe(rebuild_microscope, names='value')

shift_balance_x_slider.observe(rebuild_microscope, names='value')

shift_balance_y_slider.observe(rebuild_microscope, names='value')

tilt_balance_x_slider.observe(rebuild_microscope, names='value')

tilt_balance_y_slider.observe(rebuild_microscope, names='value')



wobble_toggle.observe(_on_wobble_toggle, names='value')

solve_shift_balance_btn.on_click(_on_solve_shift_balance)

solve_tilt_balance_btn.on_click(_on_solve_tilt_balance)

reset_deflector_btn.on_click(_on_reset_deflector)



# ── Layout ───────────────────────────────────────────────────────────



aperture_controls = widgets.VBox([

    widgets.HTML('<h4 style="margin:0 0 4px">Aperture</h4>'),

    widgets.HBox([aperture_dropdown, ap_step_dropdown]),

    widgets.HBox([ap_x_minus, ap_x_label, ap_x_plus, widgets.HTML('&nbsp;'), ap_y_minus, ap_y_label, ap_y_plus]),

], layout=widgets.Layout(border='1px solid #ddd', padding='8px', width='fit-content'))



deflector_controls = widgets.VBox([

    widgets.HTML('<h4 style="margin:0 0 4px">Double Deflector</h4>'),

    widgets.HBox([shift_x_slider, shift_y_slider]),

    widgets.HBox([tilt_x_slider, tilt_y_slider]),

    widgets.HBox([shift_balance_x_slider, shift_balance_y_slider]),

    widgets.HBox([tilt_balance_x_slider, tilt_balance_y_slider]),

    widgets.HBox([pivot_shift_x_label, pivot_shift_y_label]),

    widgets.HBox([pivot_tilt_x_label, pivot_tilt_y_label]),

    widgets.HBox([wobble_toggle, wobble_amp_slider, wobble_axis_dropdown, wobble_warning]),

    widgets.HBox([solve_shift_balance_btn, solve_tilt_balance_btn, reset_deflector_btn, solve_status_label]),

], layout=widgets.Layout(border='1px solid #ddd', padding='8px', width='fit-content'))



beam_controls = widgets.VBox([

    widgets.HTML('<h4 style="margin:8px 0 4px">Beam Slice</h4>'),

    widgets.HBox([z_slider]),

    widgets.HBox([z_minus_btn, z_plus_btn, z_step_dropdown, snap_dropdown]),

    widgets.HBox([hw_slider, gain_slider, auto_extent_toggle]),

    widgets.HBox([accum_toggle, accum_persist_toggle, accum_rays_dropdown, accum_clear_btn, accum_label]),

])



sample_controls = widgets.VBox([

    widgets.HTML('<h4 style="margin:0 0 4px">Sample</h4>'),

    widgets.HBox([sample_dropdown]),

    sample_defocus_slider,

], layout=widgets.Layout(border='1px solid #ddd', padding='8px', width='fit-content'))



controls = widgets.VBox([

    widgets.HTML('<h4 style="margin:0 0 4px">Microscope</h4>'),

    widgets.HBox([spot_slider, mag_slider, ray_count_dropdown]),

    aperture_controls,

    deflector_controls,

    beam_controls,

    sample_controls,

])

In [ ]:
# Keep UI labels consistent with the selected mode names/control names from the loaded config.
spot_slider.description = SPOT_CONTROL_LABEL
mag_slider.description = MAG_CONTROL_LABEL

In [5]:
# ── Stigmator and brightness controls ───────────────────────────────────

_stig_i_range = aux.get('stigmator_current_range_a', [-2e-3, 2e-3])
_stig_i_lo = float(min(_stig_i_range[0], _stig_i_range[1]))
_stig_i_hi = float(max(_stig_i_range[0], _stig_i_range[1]))
_stig_i_step = float(aux.get('stigmator_current_step_a', max((_stig_i_hi - _stig_i_lo) / 200.0, 1e-5)))

_cond_power_per_a = float(aux.get('condenser_stigmator_power_per_a', 400.0))
_obj_power_per_a = float(aux.get('objective_post_stigmator_power_per_a', 400.0))

_cond_ix_default_a = float(aux.get('condenser_stigmator_ix_default_a', 0.0))
_cond_iy_default_a = float(aux.get('condenser_stigmator_iy_default_a', 0.0))
_obj_ix_default_a = float(aux.get('objective_post_stigmator_ix_default_a', 0.0))
_obj_iy_default_a = float(aux.get('objective_post_stigmator_iy_default_a', 0.0))

_bright_i_range = aux.get('brightness_cl3_current_offset_range_a', [-5e-3, 5e-3])
_bright_i_lo = float(min(_bright_i_range[0], _bright_i_range[1]))
_bright_i_hi = float(max(_bright_i_range[0], _bright_i_range[1]))
_bright_i_step = float(aux.get('brightness_cl3_current_offset_step_a', max((_bright_i_hi - _bright_i_lo) / 200.0, 1e-5)))
_bright_i_default = float(aux.get('brightness_cl3_current_offset_default_a', 0.0))


def _focal_to_power(focal_m):
    f = float(focal_m)
    if not np.isfinite(f):
        return 0.0
    if abs(f) < 1e-18:
        return np.sign(f) * 1e18
    return 1.0 / f


def _power_to_focal(power_m_inv, eps=1e-12):
    p = float(power_m_inv)
    if abs(p) < eps:
        return np.inf
    return 1.0 / p


def _stigmator_focal_from_current(base_focal_m, current_a, power_per_a):
    base_power = _focal_to_power(base_focal_m)
    corrected_power = base_power + float(current_a) * float(power_per_a)
    return _power_to_focal(corrected_power)


def _fmt_focal_label(f):
    if not np.isfinite(f):
        return 'inf'
    return f"{f*1e3:.3f} mm"


def _extract_spot_value(args, kwargs):
    if 'spot_val' in kwargs:
        return float(kwargs['spot_val'])
    if len(args) > 0:
        return float(args[0])
    return float(spot_slider.value)


def _cl3_em_from_spot(spot_val):
    # Lens ordering from design curves: CL1, CL3, C_mini, Obj_prefield, ...
    return model.build_components('spot', float(spot_val))[1]


def _cl3_focal_from_offset(spot_val, offset_a):
    cl3_em = _cl3_em_from_spot(spot_val)
    effective_current = float(cl3_em.current) + float(offset_a)
    denom = float(cl3_em.Gc) * (float(cl3_em.turns) * effective_current) ** 2
    focal = np.inf if abs(denom) < 1e-30 else 1.0 / denom
    return cl3_em, effective_current, focal


def _fmt_a(v):
    return f'{float(v):+.5f} A'


def _fmt_f_m(v):
    if not np.isfinite(v):
        return 'inf'
    return f'{float(v) * 1e3:.4f} mm'


cond_stig_ix_slider = widgets.FloatSlider(
    value=float(np.clip(_cond_ix_default_a, _stig_i_lo, _stig_i_hi)),
    min=_stig_i_lo,
    max=_stig_i_hi,
    step=_stig_i_step,
    description='Cond Ix [A]',
    continuous_update=False,
    readout_format='.5f',
    layout=widgets.Layout(width='350px'),
)
cond_stig_iy_slider = widgets.FloatSlider(
    value=float(np.clip(_cond_iy_default_a, _stig_i_lo, _stig_i_hi)),
    min=_stig_i_lo,
    max=_stig_i_hi,
    step=_stig_i_step,
    description='Cond Iy [A]',
    continuous_update=False,
    readout_format='.5f',
    layout=widgets.Layout(width='350px'),
)
obj_post_stig_ix_slider = widgets.FloatSlider(
    value=float(np.clip(_obj_ix_default_a, _stig_i_lo, _stig_i_hi)),
    min=_stig_i_lo,
    max=_stig_i_hi,
    step=_stig_i_step,
    description='ObjPost Ix [A]',
    continuous_update=False,
    readout_format='.5f',
    layout=widgets.Layout(width='350px'),
)
obj_post_stig_iy_slider = widgets.FloatSlider(
    value=float(np.clip(_obj_iy_default_a, _stig_i_lo, _stig_i_hi)),
    min=_stig_i_lo,
    max=_stig_i_hi,
    step=_stig_i_step,
    description='ObjPost Iy [A]',
    continuous_update=False,
    readout_format='.5f',
    layout=widgets.Layout(width='350px'),
)

brightness_cl3_offset_slider = widgets.FloatSlider(
    value=float(np.clip(_bright_i_default, _bright_i_lo, _bright_i_hi)),
    min=_bright_i_lo,
    max=_bright_i_hi,
    step=_bright_i_step,
    description='Brightness dI CL3 [A]',
    continuous_update=False,
    readout_format='.5f',
    layout=widgets.Layout(width='450px'),
)
brightness_cl3_label = widgets.Label(layout=widgets.Layout(width='900px'))

cond_stig_focal_label = widgets.Label(layout=widgets.Layout(width='450px'))
obj_post_stig_focal_label = widgets.Label(layout=widgets.Layout(width='450px'))


def _current_to_stig_focals(cond_ix, cond_iy, obj_ix, obj_iy):
    cond_fx = _stigmator_focal_from_current(COND_STIG_FX_DEFAULT, cond_ix, _cond_power_per_a)
    cond_fy = _stigmator_focal_from_current(COND_STIG_FY_DEFAULT, cond_iy, _cond_power_per_a)
    obj_fx = _stigmator_focal_from_current(OBJ_POST_STIG_FX_DEFAULT, obj_ix, _obj_power_per_a)
    obj_fy = _stigmator_focal_from_current(OBJ_POST_STIG_FY_DEFAULT, obj_iy, _obj_power_per_a)
    return cond_fx, cond_fy, obj_fx, obj_fy


def _update_stigmator_labels(*_):
    cond_fx, cond_fy, obj_fx, obj_fy = _current_to_stig_focals(
        cond_stig_ix_slider.value,
        cond_stig_iy_slider.value,
        obj_post_stig_ix_slider.value,
        obj_post_stig_iy_slider.value,
    )
    cond_stig_focal_label.value = (
        f"Cond effective focal: fx={_fmt_focal_label(cond_fx)}, fy={_fmt_focal_label(cond_fy)}"
    )
    obj_post_stig_focal_label.value = (
        f"ObjPost effective focal: fx={_fmt_focal_label(obj_fx)}, fy={_fmt_focal_label(obj_fy)}"
    )


def _update_brightness_label(*_):
    spot_val = float(spot_slider.value)
    cl3_em, i_eff, f_eff = _cl3_focal_from_offset(spot_val, brightness_cl3_offset_slider.value)
    brightness_cl3_label.value = (
        f'CL3 current base={_fmt_a(cl3_em.current)}, offset={_fmt_a(brightness_cl3_offset_slider.value)}, '
        f'effective={_fmt_a(i_eff)} | CL3 focal base={_fmt_f_m(cl3_em.focal_length)}, effective={_fmt_f_m(f_eff)}'
    )


# Wrap build_microscope so existing rebuild/reset logic can keep calling it unchanged.
_orig_build_microscope = build_microscope


def build_microscope(*args, **kwargs):
    cond_fx, cond_fy, obj_fx, obj_fy = _current_to_stig_focals(
        cond_stig_ix_slider.value,
        cond_stig_iy_slider.value,
        obj_post_stig_ix_slider.value,
        obj_post_stig_iy_slider.value,
    )
    kwargs.setdefault('condenser_stig_fx', float(cond_fx))
    kwargs.setdefault('condenser_stig_fy', float(cond_fy))
    kwargs.setdefault('objective_post_stig_fx', float(obj_fx))
    kwargs.setdefault('objective_post_stig_fy', float(obj_fy))

    spot_val = _extract_spot_value(args, kwargs)
    offset_a = float(brightness_cl3_offset_slider.value)

    micro_dict = _orig_build_microscope(*args, **kwargs)
    cl3_em, i_eff, f_eff = _cl3_focal_from_offset(spot_val, offset_a)

    comps = list(micro_dict['components'])
    if len(comps) > 1 and isinstance(comps[1], Lens):
        old = comps[1]
        comps[1] = Lens(
            z=float(old.z),
            focal_length=float(f_eff),
            x0=float(getattr(old, 'x0', 0.0)),
            y0=float(getattr(old, 'y0', 0.0)),
        )
        micro_dict['components'] = comps

    micro_dict['brightness_cl3_offset_a'] = float(offset_a)
    micro_dict['cl3_base_current_a'] = float(cl3_em.current)
    micro_dict['cl3_effective_current_a'] = float(i_eff)
    micro_dict['cl3_effective_focal_m'] = float(f_eff)
    return micro_dict


_cache['cond_stig_ix_a'] = cond_stig_ix_slider.value
_cache['cond_stig_iy_a'] = cond_stig_iy_slider.value
_cache['obj_post_stig_ix_a'] = obj_post_stig_ix_slider.value
_cache['obj_post_stig_iy_a'] = obj_post_stig_iy_slider.value
_cache['brightness_cl3_offset_a'] = brightness_cl3_offset_slider.value


for _slider in [
    cond_stig_ix_slider,
    cond_stig_iy_slider,
    obj_post_stig_ix_slider,
    obj_post_stig_iy_slider,
]:
    _slider.observe(rebuild_microscope, names='value')
    _slider.observe(_update_stigmator_labels, names='value')

brightness_cl3_offset_slider.observe(rebuild_microscope, names='value')
brightness_cl3_offset_slider.observe(_update_brightness_label, names='value')
spot_slider.observe(_update_brightness_label, names='value')


def _reset_stigmator_controls(_):
    for _slider in [
        cond_stig_ix_slider,
        cond_stig_iy_slider,
        obj_post_stig_ix_slider,
        obj_post_stig_iy_slider,
    ]:
        _slider.unobserve(rebuild_microscope, names='value')

    cond_stig_ix_slider.value = float(np.clip(_cond_ix_default_a, _stig_i_lo, _stig_i_hi))
    cond_stig_iy_slider.value = float(np.clip(_cond_iy_default_a, _stig_i_lo, _stig_i_hi))
    obj_post_stig_ix_slider.value = float(np.clip(_obj_ix_default_a, _stig_i_lo, _stig_i_hi))
    obj_post_stig_iy_slider.value = float(np.clip(_obj_iy_default_a, _stig_i_lo, _stig_i_hi))

    for _slider in [
        cond_stig_ix_slider,
        cond_stig_iy_slider,
        obj_post_stig_ix_slider,
        obj_post_stig_iy_slider,
    ]:
        _slider.observe(rebuild_microscope, names='value')

    _update_stigmator_labels()


def _reset_brightness_control(_):
    brightness_cl3_offset_slider.unobserve(rebuild_microscope, names='value')
    brightness_cl3_offset_slider.value = float(np.clip(_bright_i_default, _bright_i_lo, _bright_i_hi))
    brightness_cl3_offset_slider.observe(rebuild_microscope, names='value')
    _update_brightness_label()


reset_deflector_btn.on_click(_reset_stigmator_controls)
reset_deflector_btn.on_click(_reset_brightness_control)

stigmator_controls = widgets.VBox([
    widgets.HTML('<h4 style="margin:0 0 4px">Stigmators (Current Controls)</h4>'),
    widgets.HTML('<div style="font-size:12px;color:#555">Small current range only; currents are mapped to weak focal-power correction (1/f).</div>'),
    widgets.HBox([cond_stig_ix_slider, cond_stig_iy_slider]),
    cond_stig_focal_label,
    widgets.HBox([obj_post_stig_ix_slider, obj_post_stig_iy_slider]),
    obj_post_stig_focal_label,
], layout=widgets.Layout(border='1px solid #ddd', padding='8px', width='fit-content'))


if not any(
    isinstance(ch, widgets.VBox)
    and len(ch.children) > 0
    and isinstance(ch.children[0], widgets.HTML)
    and ('Stigmators' in ch.children[0].value)
    for ch in controls.children
):
    _children = list(controls.children)
    _children.insert(3, stigmator_controls)
    controls.children = tuple(_children)


# Integrate brightness into the microscope header row, before Beam rays.
if len(controls.children) > 1 and isinstance(controls.children[1], widgets.HBox):
    _header_children = list(controls.children[1].children)
    if brightness_cl3_offset_slider in _header_children:
        _header_children.remove(brightness_cl3_offset_slider)
    _ray_idx = _header_children.index(ray_count_dropdown) if ray_count_dropdown in _header_children else len(_header_children)
    _header_children.insert(_ray_idx, brightness_cl3_offset_slider)
    controls.children[1].children = tuple(_header_children)

if brightness_cl3_label not in controls.children:
    _children = list(controls.children)
    _children.insert(2, brightness_cl3_label)
    controls.children = tuple(_children)


_update_stigmator_labels()
_update_brightness_label()
rebuild_microscope()

In [8]:
# ── Accumulation speed/fade tuning patch ──────────────────────────────

if 'accum_rate_dropdown' not in globals():
    accum_rate_dropdown = widgets.Dropdown(
        options=[('Fast (20 Hz)', 0.05), ('Medium (10 Hz)', 0.10), ('Slow (5 Hz)', 0.20)],
        value=0.05,
        description='Update',
        layout=widgets.Layout(width='200px'),
    )

if 'accum_fade_slider' not in globals():
    accum_fade_slider = widgets.FloatSlider(
        value=0.98,
        min=0.80,
        max=1.00,
        step=0.01,
        description='Retain',
        readout_format='.2f',
        layout=widgets.Layout(width='280px'),
    )

_accum_state.setdefault('weight', 0.0)


def _accum_status_text():
    n_frames = int(_accum_state['n_frames'])
    weight = float(_accum_state.get('weight', 0.0))
    rays_eff = int(round(weight * int(accum_rays_dropdown.value)))
    return f"{n_frames} frames · {rays_eff:,} eff rays"


def _accum_clear(*, stop_timer=True, reset_buffer=True):
    if stop_timer:
        t = _accum_state.get('timer')
        if t is not None:
            t.cancel()
            _accum_state['timer'] = None
    if reset_buffer:
        _accum_state['buffer'] = None
        _accum_state['det'] = None
        _accum_state['n_frames'] = 0
        _accum_state['weight'] = 0.0
    accum_label.value = _accum_status_text()


def _accum_tick():
    if not accum_toggle.value:
        return

    micro_now = _cache['micro']
    z_target = float(z_slider.value)
    n_batch = int(accum_rays_dropdown.value)

    hw = _compute_half_width(micro_now, z_target)
    beam_new = make_source_beam(n_batch, randomize=True)
    steps_new = propagate_steps(beam_new, micro_now['components'])
    img_new, det_new = beam_image_at_z(steps_new, micro_now['components'], z_target, hw)

    if _accum_state['buffer'] is None or _accum_state['det'] is None:
        _accum_state['buffer'] = np.asarray(img_new, dtype=np.float64)
        _accum_state['det'] = det_new
        _accum_state['n_frames'] = 1
        _accum_state['weight'] = 1.0
    else:
        fade = float(accum_fade_slider.value)
        fade = min(max(fade, 0.0), 1.0)

        det_prev = _accum_state['det']
        same_shape = tuple(getattr(det_prev, 'shape', ())) == tuple(getattr(det_new, 'shape', ()))
        same_extent = np.allclose(np.asarray(det_prev.extent, dtype=float), np.asarray(det_new.extent, dtype=float), rtol=1e-12, atol=1e-15)

        if same_shape and same_extent:
            _accum_state['buffer'] = _accum_state['buffer'] * fade + np.asarray(img_new, dtype=np.float64)
            _accum_state['weight'] = float(_accum_state.get('weight', 0.0)) * fade + 1.0
            _accum_state['n_frames'] = int(_accum_state['n_frames']) + 1
        else:
            _accum_state['buffer'] = np.asarray(img_new, dtype=np.float64)
            _accum_state['det'] = det_new
            _accum_state['n_frames'] = 1
            _accum_state['weight'] = 1.0

    accum_label.value = _accum_status_text()
    update_beam_image()

    if accum_toggle.value:
        dt = float(accum_rate_dropdown.value)
        t = threading.Timer(max(0.02, dt), _accum_tick)
        t.daemon = True
        _accum_state['timer'] = t
        t.start()


def _on_accum_params_change(_):
    accum_label.value = _accum_status_text()


def update_beam_image(*_):
    micro_now = _cache['micro']
    beam_steps_now = _cache['beam_steps']
    z_target = float(z_slider.value)
    gain = float(gain_slider.value)

    if accum_toggle.value and _accum_state['buffer'] is not None and _accum_state['det'] is not None:
        img = np.asarray(_accum_state['buffer'], dtype=np.float64)
        det = _accum_state['det']
    else:
        hw = _compute_half_width(micro_now, z_target)
        img, det = beam_image_at_z(beam_steps_now, micro_now['components'], z_target, hw)

    extent = np.asarray(det.extent) * 1e6

    if accum_toggle.value and float(_accum_state.get('weight', 0.0)) > 0.0:
        img_display = img / float(_accum_state['weight'])
    else:
        img_display = img

    scaled = img_display * gain

    z_ap_local = micro_now['z_aperture']
    show_circle = abs(z_target - z_ap_local) < 2e-3
    ap_r_um = micro_now['aperture_radius_m'] * 1e6
    ap_x_um = _cache['ap_x_um']
    ap_y_um = _cache['ap_y_um']

    with fig.batch_update():
        x0_hm, dx_hm, y0_hm, dy_hm = _extent_params(extent, img.shape)
        fig.data[IMAGE_TRACE_IDX].z = scaled.tolist()
        fig.data[IMAGE_TRACE_IDX].x0 = float(x0_hm)
        fig.data[IMAGE_TRACE_IDX].dx = float(dx_hm)
        fig.data[IMAGE_TRACE_IDX].y0 = float(y0_hm)
        fig.data[IMAGE_TRACE_IDX].dy = float(dy_hm)
        fig.data[IMAGE_TRACE_IDX].zmax = max(float(np.max(scaled)), 1e-30)
        fig.layout.xaxis.range = [float(extent[0]), float(extent[1])]
        fig.layout.yaxis.range = [float(extent[2]), float(extent[3])]

        _sync_colorbar_to_heatmap()

        fig.data[Z_LINE_IDX].y = [z_target, z_target]

        fig.data[APERTURE_CIRCLE_IDX].visible = show_circle
        if show_circle:
            fig.data[APERTURE_CIRCLE_IDX].x = (ap_x_um + ap_r_um * np.cos(theta_circle)).tolist()
            fig.data[APERTURE_CIRCLE_IDX].y = (ap_y_um + ap_r_um * np.sin(theta_circle)).tolist()


# Update beam controls layout to expose new controls.
if len(beam_controls.children) >= 5:
    beam_controls.children = (
        beam_controls.children[0],
        beam_controls.children[1],
        beam_controls.children[2],
        beam_controls.children[3],
        widgets.HBox([accum_toggle, accum_persist_toggle, accum_rays_dropdown, accum_rate_dropdown, accum_clear_btn]),
        widgets.HBox([accum_fade_slider, accum_label]),
    )

# Register parameter listeners for label updates.
accum_rays_dropdown.observe(_on_accum_params_change, names='value')
accum_rate_dropdown.observe(_on_accum_params_change, names='value')
accum_fade_slider.observe(_on_accum_params_change, names='value')

# Ensure current label and image reflect weighted accumulation semantics.
accum_label.value = _accum_status_text()
update_beam_image()

In [6]:

display(widgets.VBox([fig, controls]))


    'data': [{'colorbar': {'len': 0.6716190476190477,
                          …

In [7]:
# ── Correct relay analysis ──────────────────────────────────────────
# Fix: only include components between z_start and z_end

def abcd_between(components, labels, z_start, z_end):
    """ABCD matrix from z_start to z_end, including only components in [z_start, z_end]."""
    M = np.eye(2)
    z_prev = z_start
    for c in components:
        z_c = float(c.z)
        if z_c <= z_start + 1e-12:
            continue
        if z_c > z_end + 1e-12:
            break
        M = _abcd_free(z_c - z_prev) @ M
        z_prev = z_c
        if isinstance(c, Lens):
            M = _abcd_lens(c.focal_length) @ M
        elif isinstance(c, Stigmator):
            M = _abcd_stigmator(c.focal_length_x, c.focal_length_y) @ M
    if z_prev < z_end - 1e-12:
        M = _abcd_free(z_end - z_prev) @ M
    return M

comps  = list(_cache["micro"]["components"])
labels = list(_cache["micro"]["labels"])

z_sample   = float(comps[labels.index("Sample")].z)
z_obj_post = float(comps[labels.index("Obj_post")].z)
z_il1      = float(comps[labels.index("IL1")].z)
z_il2      = float(comps[labels.index("IL2")].z)
z_il3      = float(comps[labels.index("IL3")].z)
z_pl1      = float(comps[labels.index("PL1")].z)
z_detector = float(comps[labels.index("Detector")].z)

f_obj = float(comps[labels.index("Obj_post")].focal_length)
f_il1 = float(comps[labels.index("IL1")].focal_length)
f_il2 = float(comps[labels.index("IL2")].focal_length)
f_il3 = float(comps[labels.index("IL3")].focal_length)
f_pl1 = float(comps[labels.index("PL1")].focal_length)

# Reproduce design curves geometry
d_obj_to_il1  = z_il1 - z_obj_post
d_il1_to_il2  = z_il2 - z_il1
d_il2_to_il3  = z_il3 - z_il2
d_il3_to_pl1  = z_pl1 - z_il3
d_pl1_to_det  = z_detector - z_pl1
d_sample_to_obj = z_obj_post - z_sample

# Design curves define: d_pre_PL1_image_to_PL1 = d_pl1_to_det / M_proj
# and: d_il3_to_image = d_il3_to_pl1 - d_pre_PL1_image_to_PL1
M_proj_design = 100.0
d_pre_pl1_img_to_pl1 = d_pl1_to_det / abs(M_proj_design)
d_il3_to_image = d_il3_to_pl1 - d_pre_pl1_img_to_pl1
z_il_image = z_il3 + d_il3_to_image  # expected IL image plane

print("=== Geometry ===")
print(f"  Sample → Obj_post : {d_sample_to_obj*1e3:.3f} mm")
print(f"  Obj_post → IL1    : {d_obj_to_il1*1e3:.3f} mm")
print(f"  IL1 → IL2         : {d_il1_to_il2*1e3:.3f} mm")
print(f"  IL2 → IL3         : {d_il2_to_il3*1e3:.3f} mm")
print(f"  IL3 → PL1         : {d_il3_to_pl1*1e3:.3f} mm")
print(f"  PL1 → Detector    : {d_pl1_to_det*1e3:.3f} mm")
print(f"  IL3 → IL image    : {d_il3_to_image*1e3:.3f} mm  (z = {z_il_image*1e3:.3f} mm)")
print(f"  IL image → PL1    : {d_pre_pl1_img_to_pl1*1e3:.3f} mm")

print(f"\n=== Focal lengths (current mag setting) ===")
print(f"  Obj_post: {f_obj*1e3:.4f} mm")
print(f"  IL1:      {f_il1*1e3:.4f} mm")
print(f"  IL2:      {f_il2*1e3:.4f} mm")
print(f"  IL3:      {f_il3*1e3:.4f} mm")
print(f"  PL1:      {f_pl1*1e3:.4f} mm")

# ── Check 1: Obj_post thin-lens equation ──
u_obj = d_sample_to_obj
v_obj = 1.0 / (1.0 / f_obj - 1.0 / u_obj) if abs(u_obj) > 1e-15 else np.inf
print(f"\n=== Obj_post thin-lens check ===")
print(f"  u = {u_obj*1e3:.4f} mm,  f = {f_obj*1e3:.4f} mm")
print(f"  u < f? {u_obj < f_obj}  →  {'VIRTUAL image (u < f)' if u_obj < f_obj else 'REAL image'}")
print(f"  v = {v_obj*1e3:.4f} mm")
print(f"  For M_obj = -80: need u = f*(1+1/80) = {f_obj*(1+1/80)*1e3:.4f} mm")
print(f"  For M_obj = -80: need f < u = {u_obj*1e3:.4f} mm → f < {u_obj*1e3:.4f} mm")
print(f"  Actual f/u ratio: {f_obj/u_obj:.4f}  (need < 1.0 for real image)")

# ── Check 2: Correct B(z) sweep from sample ──
zs = np.linspace(z_sample + 0.001, z_detector + 0.01, 20_000)
Bs = np.array([float(abcd_between(comps, labels, z_sample, z)[0, 1]) for z in zs])

sign_changes = np.where(np.diff(np.sign(Bs)))[0]
print(f"\n=== B(z) = 0 crossings from sample (correct, downstream-only) ===")
image_zs = []
for idx in sign_changes:
    z0, z1 = zs[idx], zs[idx + 1]
    b0, b1 = Bs[idx], Bs[idx + 1]
    z_cross = z0 - b0 * (z1 - z0) / (b1 - b0)
    image_zs.append(z_cross)

    # Identify which lens this is after
    prev_lens = "—"
    for c, lab in zip(comps, labels):
        if (isinstance(c, Lens) or isinstance(c, Stigmator)) and float(c.z) < z_cross:
            prev_lens = lab

    # Compute magnification A at this plane
    M_at = abcd_between(comps, labels, z_sample, float(z_cross))
    A_at = float(M_at[0, 0])
    print(f"  Image at z = {z_cross*1e3:.3f} mm  (after {prev_lens:8s})  A = {A_at:.2f}")

print(f"\n  Expected IL image z = {z_il_image*1e3:.3f} mm")
print(f"  B at detector: {float(abcd_between(comps, labels, z_sample, z_detector)[0,1])*1e3:.3f} mm/rad")

# ── Check 3: IL system matrix (matching design curves definition) ──
# This is from Obj_post plane through IL1, IL2, IL3 to the IL image plane
M_il = abcd_between(comps, labels, z_obj_post, z_il_image)
A_il, B_il = float(M_il[0, 0]), float(M_il[0, 1])
print(f"\n=== IL system matrix (Obj_post → IL image at {z_il_image*1e3:.3f} mm) ===")
print(f"  A = {A_il:.6f}  (this should be M_IL target)")
print(f"  B = {B_il*1e3:.6f} mm/rad  (design needs this → 0 for imaging)")
print(f"  Full matrix:\n{M_il}")

# ── Check 4: PL1 imaging ──
# PL1 object distance = d_pre_pl1_img_to_pl1; PL1 image distance = d_pl1_to_det
u_pl1 = d_pre_pl1_img_to_pl1
v_pl1 = 1.0 / (1.0/f_pl1 - 1.0/u_pl1) if abs(u_pl1) > 1e-15 else np.inf
f_pl1_design = 1.0 / (1.0/d_pre_pl1_img_to_pl1 + 1.0/d_pl1_to_det)
print(f"\n=== PL1 check ===")
print(f"  Design requires f_PL1 = {f_pl1_design*1e3:.4f} mm")
print(f"  Actual f_PL1          = {f_pl1*1e3:.4f} mm")
print(f"  Match: {np.isclose(f_pl1, f_pl1_design, rtol=1e-3)}")

# ── B(z) plot ──
fig_bz = go.FigureWidget()
fig_bz.add_scatter(x=zs * 1e3, y=Bs * 1e3, mode="lines", name="B(z) from sample")
fig_bz.add_hline(y=0, line_dash="dash", line_color="gray")
for z_img in image_zs:
    fig_bz.add_vline(x=z_img * 1e3, line_dash="dot", line_color="red",
                     annotation_text=f"img {z_img*1e3:.1f}")
for c, lab in zip(comps, labels):
    if (isinstance(c, Lens) or isinstance(c, Stigmator)) and float(c.z) > z_sample:
        fig_bz.add_vline(x=float(c.z) * 1e3, line_dash="dash", line_color="blue",
                         annotation_text=lab, annotation_position="bottom right")
fig_bz.add_vline(x=z_il_image * 1e3, line_dash="dashdot", line_color="green",
                 annotation_text="IL image (design)")
fig_bz.update_layout(
    title="B(z) from sample — correct (downstream-only components)",
    xaxis_title="z (mm)", yaxis_title="B (mm/rad)", width=1100, height=450,
)
fig_bz


=== Geometry ===
  Sample → Obj_post : 2.300 mm
  Obj_post → IL1    : 78.083 mm
  IL1 → IL2         : 61.233 mm
  IL2 → IL3         : 68.851 mm
  IL3 → PL1         : 47.449 mm
  PL1 → Detector    : 325.000 mm
  IL3 → IL image    : 44.199 mm  (z = 579.666 mm)
  IL image → PL1    : 3.250 mm

=== Focal lengths (current mag setting) ===
  Obj_post: 2.2258 mm
  IL1:      7.8536 mm
  IL2:      2.2017 mm
  IL3:      25.7740 mm
  PL1:      3.2178 mm

=== Obj_post thin-lens check ===
  u = 2.3000 mm,  f = 2.2258 mm
  u < f? False  →  REAL image
  v = 69.0000 mm
  For M_obj = -80: need u = f*(1+1/80) = 2.2536 mm
  For M_obj = -80: need f < u = 2.3000 mm → f < 2.3000 mm
  Actual f/u ratio: 0.9677  (need < 1.0 for real image)

=== B(z) = 0 crossings from sample (correct, downstream-only) ===
  Image at z = 396.300 mm  (after ObjPost_Stig)  A = -30.00
  Image at z = 463.409 mm  (after IL1     )  A = 191.65
  Image at z = 473.639 mm  (after IL2     )  A = -419.66
  Image at z = 579.666 mm  (after IL

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'B(z) from sample',
              'type': 'scatter',
              'uid': 'a988b9f6-b5bd-426b-8cd0-cfeabea3f68d',
              'x': {'bdata': ('AAAAAABgdEARUf86eWB0QCCi/nXyYH' ... 'zaroxAjTBPWhevjEAW2c73U6+MQA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('BAAAAAAA8D9iEFH/OnnwP8Yfov518v' ... 'NQtzW/Nt8fo93HNb/sGvryatg1vw=='),
                    'dtype': 'f8'}}],
    'layout': {'annotations': [{'showarrow': False,
                                'text': 'img 396.3',
                                'x': np.float64(396.2999999999785),
                                'xanchor': 'left',
                                'xref': 'x',
                                'y': 1,
                                'yanchor': 'top',
                                'yref': 'y domain'},
                               {'showarrow': False,
                                'text': 'img 463.4',
                       